# APC-Vis · Qwen2.5-VL-7B · Full phần 2 · Object Facing

Kaggle **GPU T4 ×2**, Internet và Secret **HF_token**. Chọn **Run All**.

Chạy APC-Vis trên 692 lượt. Các hoán vị đáp án của mỗi nhóm nằm cùng một notebook.

Cấu hình model, phân bổ GPU và trace giữ theo lượt Kaggle dispatch v3.
Output có thư mục và tên ZIP riêng cho lượt này.


In [ ]:
from pathlib import Path
import os,sys,json,subprocess,time,hashlib,shutil,base64,io,zipfile
WORK=Path('/kaggle/working/apc_qwen7b');WORK.mkdir(parents=True,exist_ok=True)
token=os.environ.get('HF_TOKEN')
if not token:
    try:
        from kaggle_secrets import UserSecretsClient
        token=UserSecretsClient().get_secret('HF_token')
    except Exception:
        token=None
if token:os.environ['HF_TOKEN']=token.strip()
else:print('HF_token chưa đặt; dùng quyền truy cập public. Có thể thêm Secret HF_token nếu bị giới hạn tải.')
del token
os.environ['TOKENIZERS_PARALLELISM']='false';os.environ['HF_HUB_DISABLE_TELEMETRY']='1';os.environ['HF_HUB_DISABLE_XET']='1'
SNAPSHOT_SHA='9059ed359baf89b49675ed3e0407f843b3ff28de78e5cae1c870adf79f44a339'
REPO=WORK/('repo_'+SNAPSHOT_SHA[:12]);REPO.mkdir(exist_ok=True)
raw=base64.b64decode('...')
assert hashlib.sha256(raw).hexdigest()==SNAPSHOT_SHA
with zipfile.ZipFile(io.BytesIO(raw)) as z:
    for name in z.namelist():
        target=(REPO/name).resolve();target.relative_to(REPO.resolve())
    z.extractall(REPO)
SCRIPTS=WORK/'scripts';SCRIPTS.mkdir(exist_ok=True)
(SCRIPTS/'apc_common.py').write_text('"""Shared dataset/parser/state helpers; no model imports needed for offline validation."""\nimport hashlib,json,os,re,time,unicodedata\nfrom pathlib import Path\nPOLICY=\'strict_letter_or_unique_exact_choice_v1\'\ndef parse_option(text,choices):\n    def norm(x):return re.sub(r\'\\s+\',\' \',unicodedata.normalize(\'NFKC\',x).strip()).casefold()\n    match=re.fullmatch(r\'\\s*(?:(?:the\\s+)?answer\\s*(?:is|:)\\s*)?[\\(\\[]?([A-D])[\\)\\]]?[.!]?\\s*\',text,re.I)\n    strict=match.group(1).upper() if match and match.group(1).upper() in choices else None\n    exact=[k for k,v in choices.items() if norm(text) in (norm(v),norm(f\'{k}. {v}\'),norm(f\'{k}) {v}\'))]\n    return strict if strict else exact[0] if len(exact)==1 else None\n\ndef atomic_json(path,obj):\n    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True);tmp=path.with_suffix(\'.tmp\')\n    tmp.write_text(json.dumps(obj,ensure_ascii=False,indent=2),encoding=\'utf-8\');tmp.replace(path)\n\ndef read_jsonl(path,repair=False):\n    path=Path(path);rows={}\n    if not path.exists():return rows\n    with path.open(\'rb+\' if repair else \'rb\') as f:\n        while True:\n            start=f.tell();line=f.readline()\n            if not line:break\n            try:r=json.loads(line)\n            except (ValueError,UnicodeDecodeError):\n                if repair and not line.endswith(b\'\\n\') and not f.read():f.truncate(start);break\n                raise RuntimeError(\'Corrupt JSONL \'+str(path)+\' byte \'+str(start)) from None\n            rows[r[\'sample_id\']]=r\n            if repair and not line.endswith(b\'\\n\'):f.seek(0,2);f.write(b\'\\n\')\n    return rows\n\nPAPER_TASKS={\'orientation_on_the_left\':\'left_right\',\'orientation_in_front_of\':\'visibility\',\'multi_object_facing\':\'facing\'}\n\ndef paper_sample(row):\n    row=dict(row)\n    row[\'paper_task\']=PAPER_TASKS[row[\'category\']]\n    row[\'source_question\']=row[\'question\'];row[\'source_choices\']=dict(row[\'choices\'])\n    row[\'question_transform\']=\'unchanged\'\n    if row[\'category\']==\'orientation_in_front_of\':\n        match=re.fullmatch(r\'(.*?facing where it is facing, )is (.+?) in front of me or behind me\\?\',row[\'question\'],re.I)\n        if not match:raise ValueError(\'Unrecognized front/behind template for \'+row[\'sample_id\'])\n        mapping={\'in front of\':\'yes\',\'behind\':\'no\'}\n        if set(row[\'choices\'].values())!=set(mapping):raise ValueError(\'Unexpected front/behind options\')\n        row[\'question\']=match.group(1)+\'is \'+match.group(2)+\' visible or not?\'\n        row[\'choices\']={key:mapping[value] for key,value in row[\'choices\'].items()}\n        row[\'question_transform\']=\'front_behind_to_visibility_yes_no_v1\'\n        # Option positions and correct option letter are preserved, including each permutation.\n    return row\n\ndef choose_samples(rows,cfg):\n    scope=cfg.get(\'scope\')\n    if scope in (\'paper_tasks_original\',\'paper_tasks_visibility\'):\n        rows=[r for r in rows if r[\'category\'] in PAPER_TASKS]\n        if scope==\'paper_tasks_visibility\':rows=[paper_sample(r) for r in rows]\n    elif scope!=\'all_original\':raise ValueError(\'Unsupported scope\')\n    if cfg.get(\'limit\') is not None:\n        groups=set(r[\'circular_group\'] for r in rows[:cfg[\'limit\']]);rows=[r for r in rows if r[\'circular_group\'] in groups]\n    return rows\n',encoding='utf-8')
(SCRIPTS/'apc_kaggle_patches.py').write_text('"""Apply auditable Kaggle compatibility fixes to a copied APC snapshot, not the checkout."""\nimport ast,hashlib,json,os\nfrom pathlib import Path\n\ndef patch_repo(repo,assets):\n    repo=Path(repo);assets=Path(assets);changes=[]\n    def edit(rel,fn):\n        p=repo/rel\n        backup=p.with_name(p.name+\'.apc_original\')\n        if not backup.exists():backup.write_text(p.read_text(),encoding=\'utf-8\')\n        old=backup.read_text();new=fn(old)\n        if new!=old:\n            ast.parse(new);p.write_text(new,encoding=\'utf-8\')\n            changes.append(dict(file=rel,before_sha=hashlib.sha256(old.encode()).hexdigest(),after_sha=hashlib.sha256(new.encode()).hexdigest()))\n    def detection(s):\n        s=s.replace(\'load_model(config_path, config.detection.ckpt_path)\',\'load_model(config_path, config.detection.ckpt_path, device=str(self.device))\')\n        needle=\'            text_threshold=self.config.detection.text_threshold,\\n\'\n        assert needle in s;s=s.replace(needle,needle+\'            device=str(self.device),\\n\')\n        # GroundingDINO\'s custom CUDA attention uses getCurrentCUDAStream()\n        # without a device guard. Match the current GPU to its input tensors.\n        tree=ast.parse(s)\n        method=next(n for n in ast.walk(tree) if isinstance(n,ast.FunctionDef) and n.name==\'run_detection\')\n        call=next(n for n in ast.walk(method) if isinstance(n,ast.Assign) and isinstance(n.value,ast.Call) and isinstance(n.value.func,ast.Name) and n.value.func.id==\'predict\')\n        lines=s.splitlines(keepends=True)\n        block=lines[call.lineno-1:call.end_lineno]\n        lines[call.lineno-1:call.end_lineno]=[\'        with torch.cuda.device(self.device):\\n\']+[\'    \'+line for line in block]\n        s=\'\'.join(lines)\n        return s\n    edit(\'apc/vision_modules/detection.py\',detection)\n    def orientation(s):\n        s=s.replace(\'from .src.orient_anything.render import render, Model\',\'\')\n        needle=\'        axis_model = Model(axis_obj_path, texture_filename=axis_texture_path)\'\n        s=s.replace(needle,\'        from .src.orient_anything.render import render, Model\\n\'+needle)\n        tree=ast.parse(s)\n        assignment=next(n for n in ast.walk(tree) if isinstance(n,ast.Assign) and any(isinstance(t,ast.Name) and t.id==\'orientation_ckpt_path\' for t in n.targets))\n        s=s.replace(ast.get_source_segment(s,assignment),\'orientation_ckpt_path = str(config.orientation.model_path)\',1)\n        s=s.replace(\'"facebook/dinov2-large",\',\'str(config.orientation.processor_path),\')\n        return s\n    edit(\'apc/vision_modules/orientation.py\',orientation)\n    def renderer(s):\n        s=s.replace("        if apc_args[\'render_whole_scene\']:","        z_trans = 0.0  # No backward shift needed when all objects are already in front.\\n        if apc_args[\'render_whole_scene\']:")\n        s=s.replace(\'        # Normalize positions\',\'        orientations = [orientations[i] for i in positions_ori_idx]\\n        colors = [colors[i] for i in positions_ori_idx]\\n\\n        # Normalize positions\')\n        s=s.replace(\'            xy_scale = grid_size / max_xy\',\'            xy_scale = grid_size / max(max_xy, 1e-6)\')\n        s=s.replace("            z_list = [pos[2] for pos in positions]\\n            max_z_obj_idx", "            if not positions:\\n                raise ValueError(\'No objects to render after excluding reference viewer\')\\n            z_list = [pos[2] for pos in positions]\\n            max_z_obj_idx",1)\n        return s\n    edit(\'apc/renderer.py\',renderer)\n    # Keep the mask statistics/geometry method; only remove a slow Python pixel loop.\n    def depth(s):\n        a=s.index(\'        depth_values = []\',s.index(\'def unproject_to_3D\'))\n        b=s.index(\'        # Find the most frequent depth\',a)\n        s=s[:a]+"        depth_values = depth_unproj[ys,xs]\\n        if depth_values.size == 0:\\n            raise ValueError(\'Empty segmentation mask; no 3D points\')\\n\\n"+s[b:]\n        a=s.index(\'        points_filtered = []\',s.index(\'def unproject_to_3D\'))\n        b=s.index(\'        # Compute final 3D coordinates\',a)\n        s=s[:a]+"        z=depth_unproj[ys,xs][depth_values_filtered]\\n        x=(xs[depth_values_filtered]-K[0,2])*z/K[0,0]\\n        y=(ys[depth_values_filtered]-K[1,2])*z/K[1,1]\\n        points_filtered=np.column_stack((x,-y,-z))\\n        if not len(points_filtered) or not np.isfinite(points_filtered).all():\\n            raise ValueError(\'Empty/nonfinite filtered 3D points\')\\n\\n"+s[b:]\n        return s\n    edit(\'apc/vision_modules/depth.py\',depth)\n    # External sources exist after HTTPS cloning.\n    edit(\'apc/vision_modules/src/orient_anything/paths.py\',lambda s:s.replace(\'"facebook/dinov2-large"\',repr(str(assets/\'dinov2_large\'))))\n    edit(\'apc/vision_modules/src/orient_anything/vision_tower.py\',lambda s:s.replace(\'bool_masked_pos: Optional[torch.Tensor] = None) -> torch.Tensor:\', \'bool_masked_pos: Optional[torch.Tensor] = None, **kwargs) -> torch.Tensor:\').replace(\'embeddings = self.dropout(embeddings)\',"embeddings = getattr(self, \'dropout\', torch.nn.Identity())(embeddings)"))\n    edit(\'apc/vision_modules/src/GroundingDINO/groundingdino/config/GroundingDINO_SwinT_OGC.py\',lambda s:s.replace(\'"bert-base-uncased"\',repr(str(assets/\'bert_base_uncased\'))).replace("\'bert-base-uncased\'",repr(str(assets/\'bert_base_uncased\'))))\n    return changes\n\nif __name__==\'__main__\':\n    import sys\n    changes=patch_repo(sys.argv[1],sys.argv[2]);Path(sys.argv[3]).write_text(json.dumps(changes,indent=2),encoding=\'utf-8\')\n',encoding='utf-8')
(SCRIPTS/'apc_install_native.py').write_text('"""Install native vision dependencies into the isolated Kaggle venv."""\nimport hashlib,json,os,shutil,subprocess,sys,time\nfrom pathlib import Path\ncfg=json.loads(Path(sys.argv[1]).read_text());repo=Path(cfg[\'repo\']);work=Path(cfg[\'work\']);py=sys.executable\nearly_nvcc=shutil.which(\'nvcc\')\nif not early_nvcc:raise RuntimeError(\'nvcc is missing. Native APC vision setup needs a CUDA 12 development toolkit, not only an NVIDIA driver.\')\nearly_version=subprocess.check_output([early_nvcc,\'--version\'],text=True)\nif \'release 12.\' not in early_version:raise RuntimeError(\'Use a CUDA 12 toolkit with the cu124 Torch wheel. Found: \'+early_version)\nsrc=repo/\'apc/vision_modules/src\';src.mkdir(parents=True,exist_ok=True)\nlock_path=work/\'source_lock.json\';lock=json.loads(lock_path.read_text()) if lock_path.exists() else {}\nrepos={\n \'GroundingDINO\':(\'https://github.com/IDEA-Research/GroundingDINO.git\',\'main\'),\n \'ml-depth-pro\':(\'https://github.com/apple/ml-depth-pro.git\',\'main\'),\n \'orient_anything\':(\'https://github.com/SpatialVision/Orient-Anything.git\',\'main\'),\n \'omni3d\':(\'https://github.com/facebookresearch/omni3d.git\',\'main\'),\n \'detectron2\':(\'https://github.com/facebookresearch/detectron2.git\',\'main\'),\n \'pytorch3d\':(\'https://github.com/facebookresearch/pytorch3d.git\',\'V0.7.8\'),\n \'segment-anything\':(\'https://github.com/facebookresearch/segment-anything.git\',\'main\')}\ndef run(cmd,**kw):\n print(\'RUN\',cmd[:5],flush=True);subprocess.run(cmd,check=True,**kw)\nfor name,(url,ref) in repos.items():\n path=src/name\n if not (path/\'.git\').exists():\n  check=subprocess.run([\'git\',\'ls-remote\',\'--exit-code\',url,\'refs/heads/\'+ref,\'refs/tags/\'+ref],capture_output=True,text=True)\n  if check.returncode:\n   raise RuntimeError(\'Git ref missing or unreachable (case-sensitive): \'+ref+\' in \'+url+\'; \'+check.stderr.strip())\n  run([\'git\',\'clone\',\'--depth\',\'1\',\'--branch\',ref,url,str(path)])\n sha=subprocess.check_output([\'git\',\'rev-parse\',\'HEAD\'],cwd=path,text=True).strip()\n if name in lock and lock[name][\'sha\']!=sha:raise RuntimeError(\'Source revision changed: \'+name)\n lock[name]=dict(url=url,sha=sha,requested_ref=ref)\n lock_path.write_text(json.dumps(lock,indent=2),encoding=\'utf-8\')\n# Do not install Orient-Anything requirements: they downgrade transformers to 4.38.\npackages=[\'pip==25.0.1\',\'setuptools==75.3.0\',\'wheel\',\'ninja\',\'cython==0.29.37\',\'numpy==1.26.4\',\'scipy==1.11.4\',\n \'matplotlib==3.8.4\',\'python-box==7.3.2\',\'trimesh==4.7.4\',\'open3d==0.19.0\',\'pyglet==1.5.27\',\n \'opencv-python==4.10.0.84\',\'timm==1.0.22\',\'pillow-heif==0.18.0\',\'fvcore==0.1.5.post20221221\',\n \'iopath==0.1.10\',\'yacs\',\'cloudpickle\',\'termcolor\',\'pycocotools\',\'hydra-core==1.3.2\',\'omegaconf==2.3.0\',\n \'shapely==2.0.6\',\'rtree\',\'addict\',\'yapf==0.40.2\',\'supervision==0.22.0\',\'qwen-vl-utils==0.0.14\',\'av==13.1.0\']\nuv=cfg[\'uv\'];run([uv,\'pip\',\'install\',\'--python\',py,\'--no-cache\',*packages])\n# System display and compiler tools. No hardcoded DISPLAY=:1 from run_APC.py.\napt=[\'apt-get\'] if os.geteuid()==0 else [\'sudo\',\'-n\',\'apt-get\']\nrun(apt+[\'update\',\'-qq\']);run(apt+[\'install\',\'-y\',\'-qq\',\'xvfb\',\'xauth\',\'libgl1\',\'libglu1-mesa\',\'libgl1-mesa-dri\',\'build-essential\',\'ninja-build\'])\nimport torch\nnvcc=shutil.which(\'nvcc\')\nif not nvcc:raise RuntimeError(\'GroundingDINO CUDA build requires nvcc. Attach/select a Kaggle runtime with a CUDA 12 toolkit; driver alone is insufficient.\')\nversion=subprocess.check_output([nvcc,\'--version\'],text=True)\nif \'release 12.\' not in version:raise RuntimeError(\'CUDA toolkit must be major 12 for the cu124 wheel. Found: \'+version)\nenv=os.environ.copy();env[\'CUDA_HOME\']=str(Path(nvcc).resolve().parent.parent);env[\'FORCE_CUDA\']=\'1\';env[\'MAX_JOBS\']=\'2\'\nenv[\'TORCH_CUDA_ARCH_LIST\']=\';\'.join(sorted({\'.\'.join(map(str,torch.cuda.get_device_capability(i))) for i in range(torch.cuda.device_count())}))\n# Build with the already installed torch, without a second isolated copy of torch/CUDA.\nfor name in (\'segment-anything\',\'ml-depth-pro\',\'detectron2\',\'pytorch3d\',\'GroundingDINO\'):\n marker=work/(\'installed_\'+name+\'.json\')\n identity=dict(source_sha=lock[name][\'sha\'],torch=torch.__version__,python=sys.version.split()[0],cuda_compiler=version.strip())\n if marker.exists() and json.loads(marker.read_text())==identity:continue\n # A failed wheel build can leave its dist-info directory in bdist.*.\n # Remove packaging staging only, retaining compiled objects in build/temp.*.\n build_root=(src/name/\'build\').resolve()\n build_root.relative_to((src/name).resolve())\n if build_root.is_dir():\n  for staging in build_root.glob(\'bdist.*\'):\n   resolved=staging.resolve()\n   if staging.is_symlink() or resolved.parent!=build_root:\n    raise RuntimeError(\'Unexpected wheel staging path: \'+str(staging))\n   if resolved.is_dir():\n    print(\'Cleaning previous wheel staging:\',resolved,flush=True)\n    shutil.rmtree(resolved)\n run([uv,\'pip\',\'install\',\'--python\',py,\'--no-cache\',\'--no-build-isolation\',\'--no-deps\']+([\'-e\'] if name in (\'GroundingDINO\',\'ml-depth-pro\') else [])+[str(src/name)],env=env)\n marker.write_text(json.dumps(identity),encoding=\'utf-8\')\nprint(\'Native dependencies installed. Sources pinned in source_lock.json.\',flush=True)\n',encoding='utf-8')
(SCRIPTS/'apc_prepare_assets.py').write_text('"""Download once or reuse a read-only Kaggle asset input; never duplicate weight caches."""\nimport hashlib,json,os,shutil,sys,time,urllib.request\nfrom http.client import HTTPException\nfrom pathlib import Path\nfrom huggingface_hub import HfApi,hf_hub_download,snapshot_download\ncfg=json.loads(Path(sys.argv[1]).read_text());root=Path(cfg[\'asset_input\'] or cfg[\'asset_root\']);root.mkdir(parents=True,exist_ok=True)\nmanifest_file=root/\'assets_manifest.json\';old=json.loads(manifest_file.read_text()) if manifest_file.exists() else {}\nif cfg[\'asset_input\']:\n    required=[\'qwen_model/config.json\',\'qwen_model/preprocessor_config.json\',\'dinov2_large/config.json\',\'bert_base_uncased/config.json\',\n       \'checkpoints/sam_vit_h_4b8939.pth\',\'checkpoints/groundingdino_swint_ogc.pth\',\'checkpoints/depth_pro.pt\',\'checkpoints/orient_dino_weight.pt\']\n    missing=[s for s in required if not (root/s).is_file()]\n    for model in (\'qwen_model\',\'dinov2_large\',\'bert_base_uncased\'):\n        if not list((root/model).glob(\'*.safetensors\')):missing.append(model+\'/*.safetensors\')\n    if missing or not old:raise RuntimeError(\'ASSET_INPUT must be the prepared asset bundle root with assets_manifest.json. Missing \'+str(missing))\n    if old.get(\'model_id\')!=cfg[\'model_id\']:raise RuntimeError(\'Wrong Qwen checkpoint asset bundle.\')\n    Path(sys.argv[2]).write_text(json.dumps(dict(old,root=str(root.resolve())),indent=2),encoding=\'utf-8\')\n    print(\'Using existing asset bundle:\',root);raise SystemExit(0)\napi=HfApi();snapshots={}\nmodels={\'qwen_model\':cfg[\'model_id\'],\'dinov2_large\':\'facebook/dinov2-large\',\'bert_base_uncased\':\'google-bert/bert-base-uncased\'}\nfor name,repo in models.items():\n    previous=old.get(\'snapshots\',{}).get(name,{})\n    revision=previous.get(\'sha\') if previous.get(\'repo\')==repo else cfg[\'model_revision\'] if name==\'qwen_model\' else \'main\'\n    info=api.model_info(repo,revision=revision,files_metadata=True)\n    wanted=[f for f in info.siblings if \'/\' not in f.rfilename and (f.rfilename.endswith((\'.json\',\'.safetensors\',\'.txt\',\'.model\',\'.jinja\')))]\n    remaining=sum(f.size or 0 for f in wanted if not (root/name/f.rfilename).is_file() or (root/name/f.rfilename).stat().st_size!=f.size)\n    if shutil.disk_usage(root).free<remaining+3*2**30:raise RuntimeError(\'Asset storage lacks space. Use a prepared Kaggle ASSET_INPUT; need \'+str(round(remaining/2**30,2))+\' GiB + 3 GiB reserve for \'+repo)\n    print(\'Download/cache\',repo,\'SHA\',info.sha,\'remaining GiB\',round(remaining/2**30,2),flush=True)\n    for f in sorted(wanted,key=lambda f:f.rfilename.endswith(\'.safetensors\')):\n        hf_hub_download(repo,f.rfilename,revision=info.sha,local_dir=str(root/name))\n        if f.size and (root/name/f.rfilename).stat().st_size!=f.size:raise RuntimeError(\'Truncated asset: \'+f.rfilename)\n    if not list((root/name).glob(\'*.safetensors\')):raise RuntimeError(\'No safetensors weights in \'+repo+\'; stop before model initialization.\')\n    snapshots[name]=dict(repo=repo,sha=info.sha)\nckpt=root/\'checkpoints\';ckpt.mkdir(exist_ok=True)\nurls={\n \'sam_vit_h_4b8939.pth\':\'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth\',\n \'groundingdino_swint_ogc.pth\':\'https://github.com/IDEA-Research/GroundingDINO/releases/download/v0.1.0-alpha/groundingdino_swint_ogc.pth\',\n \'depth_pro.pt\':\'https://ml-site.cdn-apple.com/models/depth-pro/depth_pro.pt\'}\ndef download_checkpoint(url,target):\n    part=target.with_suffix(\'.part\')\n    for attempt in range(1,7):\n        try:\n            offset=part.stat().st_size if part.exists() else 0\n            headers={\'Accept-Encoding\':\'identity\'}\n            if offset:headers[\'Range\']=f\'bytes={offset}-\'\n            print(f\'Downloading {target.name}: attempt {attempt}/6, resume {offset/2**20:.0f} MiB\',flush=True)\n            with urllib.request.urlopen(urllib.request.Request(url,headers=headers),timeout=120) as response:\n                resumed=response.status==206\n                if resumed:\n                    content_range=response.headers.get(\'Content-Range\',\'\')\n                    if not content_range.startswith(f\'bytes {offset}-\'):raise OSError(\'Unexpected Content-Range: \'+content_range)\n                    total=int(content_range.split(\'/\')[-1])\n                else:\n                    offset=0 # Server ignored Range: overwrite, never append a full response.\n                    total=int(response.headers.get(\'Content-Length\',0))\n                if total and shutil.disk_usage(target.parent).free<total-offset+3*2**30:\n                    raise RuntimeError(\'Not enough disk for \'+target.name+\'; use ASSET_INPUT.\')\n                with part.open(\'ab\' if resumed else \'wb\') as f:\n                    while True:\n                        block=response.read(1024*1024)\n                        if not block:break\n                        f.write(block)\n                size=part.stat().st_size\n                if size<1000000 or total and size!=total:raise OSError(f\'Truncated download: {size}/{total} bytes\')\n            part.replace(target)\n            return\n        except (OSError,HTTPException,ValueError) as exc:\n            if attempt==6:raise RuntimeError(f\'Failed downloading {target.name}; partial file kept at {part}: {exc}\') from exc\n            print(f\'{target.name}: {type(exc).__name__}: {exc}; retrying...\',flush=True)\n            time.sleep(min(5*attempt,30))\n\nchecks={}\nfor name,url in urls.items():\n    target=ckpt/name;expected=old.get(\'files\',{}).get(name,{})\n    valid=target.is_file() and target.stat().st_size==expected.get(\'bytes\')\n    # Older runs wrote the manifest only after all downloads. Reuse their complete files.\n    if target.is_file() and not expected:\n        try:\n            with urllib.request.urlopen(urllib.request.Request(url,method=\'HEAD\'),timeout=60) as response:\n                valid=target.stat().st_size==int(response.headers.get(\'Content-Length\',0)) and target.stat().st_size>1000000\n        except OSError:pass\n    if not valid:\n        download_checkpoint(url,target)\n    else:print(\'Reusing\',name,flush=True)\n    digest=hashlib.sha256()\n    with target.open(\'rb\') as f:\n        for block in iter(lambda:f.read(16*1024*1024),b\'\'):digest.update(block)\n    if valid and expected.get(\'sha256\') and digest.hexdigest()!=expected[\'sha256\']:raise RuntimeError(\'Asset checksum mismatch \'+name)\n    checks[name]=dict(url=url,bytes=target.stat().st_size,sha256=digest.hexdigest())\n    # Save completed checkpoint metadata even if a later download fails.\n    old.update(model_id=cfg[\'model_id\'],snapshots=snapshots,files=checks)\n    manifest_file.write_text(json.dumps(old,indent=2),encoding=\'utf-8\')\nori_revision=old.get(\'orientation_sha\',\'main\');ori_info=api.model_info(\'Viglong/Orient-Anything\',revision=ori_revision)\nori_path=hf_hub_download(\'Viglong/Orient-Anything\',\'croplargeEX2/dino_weight.pt\',revision=ori_info.sha,local_dir=str(root/\'orientation_download\'))\n# Symlink rather than make another full checkpoint copy.\ntarget=ckpt/\'orient_dino_weight.pt\'\nif target.is_symlink() and target.resolve()!=Path(ori_path).resolve():target.unlink()\nif not target.exists():target.symlink_to(Path(ori_path).resolve())\nmanifest=dict(model_id=cfg[\'model_id\'],snapshots=snapshots,orientation_sha=ori_info.sha,files=checks,root=str(root.resolve()))\nmanifest_file.write_text(json.dumps(manifest,indent=2),encoding=\'utf-8\');Path(sys.argv[2]).write_text(json.dumps(manifest,indent=2),encoding=\'utf-8\')\nprint(\'Assets ready:\',root,flush=True)\n',encoding='utf-8')
(SCRIPTS/'apc_qwen_adapter.py').write_text('"""Qwen 7B interface compatible with the APC multi-call pipeline."""\nimport time\nimport torch\nfrom transformers import Qwen2_5_VLForConditionalGeneration,AutoProcessor\nfrom qwen_vl_utils import process_vision_info\nfrom apc.vlms.vlm_base import VLMBase\n\nclass KaggleQwen(VLMBase):\n    def __init__(self,config,device=\'cuda:0\'):\n        super().__init__(device);self.config=config;self.device=device;self.calls=[];self.callback=None\n        self.load_model(config,device)\n    def load_model(self, config, device=\'cuda:0\'):\n        from pathlib import Path\n        import json\n        from apc_kaggle_memory import make_qwen_memory_plan, verify_loaded_partition\n\n        dtype = getattr(torch, config.dtype)\n        if config.dtype == \'bfloat16\' and not torch.cuda.is_bf16_supported(including_emulation=False):\n            raise RuntimeError(\'Native BF16 unavailable on T4; select float16.\')\n        report_dir = Path(config.memory_report_dir)\n        report_dir.mkdir(parents=True, exist_ok=True)\n        try:\n            device_map, report = make_qwen_memory_plan(\n                config.pretrained_model, dtype,\n                activation_reserve_gib=config.activation_reserve_gib,\n                vision_reserve_gib=config.vision_reserve_gib,\n            )\n        except Exception as exc:\n            (report_dir/\'memory_plan.json\').write_text(json.dumps(dict(error=str(exc)), indent=2))\n            raise\n        (report_dir/\'memory_plan.json\').write_text(json.dumps(report, indent=2))\n        print(\'Qwen memory plan:\', json.dumps({k:v for k,v in report.items() if k != \'device_map\'}), flush=True)\n        self.vlm_model, loading = Qwen2_5_VLForConditionalGeneration.from_pretrained(\n            config.pretrained_model, dtype=dtype, device_map=device_map,\n            attn_implementation=\'sdpa\', local_files_only=True,\n            low_cpu_mem_usage=True, output_loading_info=True,\n        )\n        if any(loading.get(k) for k in (\'missing_keys\',\'unexpected_keys\',\'mismatched_keys\',\'error_msgs\')):\n            raise RuntimeError(\'Qwen checkpoint did not load cleanly: \'+str(loading))\n        self.vlm_model.eval()\n        self.dtype = dtype\n        self.device = str(self.vlm_model.get_input_embeddings().weight.device)\n        actual_map = self.vlm_model.hf_device_map\n        (report_dir/\'model_device_map.json\').write_text(json.dumps(actual_map, default=str, indent=2))\n        if any(str(value) not in (\'0\',\'1\',\'cuda:0\',\'cuda:1\') for value in actual_map.values()):\n            raise RuntimeError(\'Qwen must remain entirely on the two GPUs. Inspect model_device_map.json.\')\n        if any(parameter.device.type == \'meta\' for parameter in self.vlm_model.parameters()):\n            raise RuntimeError(\'Qwen contains unloaded meta parameters; stop before inference.\')\n        placement = verify_loaded_partition(self.vlm_model, device_map)\n        placement[\'memory_allocated_gib\'] = [torch.cuda.memory_allocated(i)/2**30 for i in (0,1)]\n        (report_dir/\'qwen_placement.json\').write_text(json.dumps(placement, indent=2))\n        print(\'Qwen actual placement PASS:\', json.dumps(placement), flush=True)\n        self.processor = AutoProcessor.from_pretrained(\n            config.pretrained_model, local_files_only=True,\n            **{key:getattr(config,key) for key in (\'min_pixels\',\'max_pixels\') if getattr(config,key) is not None},\n        )\n    def generate_response(self,prompt,image=None,max_new_tokens=None,do_sample=False,temperature=0.0):\n        content=([{\'type\':\'image\',\'image\':image}] if image is not None else [])+[{\'type\':\'text\',\'text\':prompt}]\n        return self.process_messages([dict(role=\'user\',content=content)],max_new_tokens,do_sample,temperature)\n    def process_messages(self,messages,max_new_tokens=None,do_sample=False,temperature=0.0):\n        if getattr(self, \'before_generate\', None):self.before_generate()\n        started=time.monotonic()\n        import os,json\n        from pathlib import Path\n        folder=Path(os.environ[\'APC_TRACE_DIR\'])/\'vlm\'\n        folder.mkdir(exist_ok=True)\n        index=len(self.calls)+1\n        captured=[]\n        for message_index,message in enumerate(messages):\n            for content_index,content in enumerate(message.get(\'content\',[])):\n                if isinstance(content,dict) and content.get(\'type\')==\'image\':\n                    value=content[\'image\']\n                    image_file=folder/f\'{index:03d}_{message_index}_{content_index}.png\'\n                    if hasattr(value,\'save\'):\n                        value.save(image_file)\n                        captured.append(image_file.name)\n        # Save before generation so a failed call still has its prompt.\n\n        if self.callback:self.callback(\'vlm_call\',call=len(self.calls)+1)\n        text=self.processor.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)\n        has_image=any(isinstance(c,dict) and c.get(\'type\')==\'image\' for m in messages for c in m.get(\'content\',[]))\n        images,videos=process_vision_info(messages) if has_image else (None,None)\n        batch=self.processor(text=[text],images=images,videos=videos,padding=True,return_tensors=\'pt\')\n        length=int(batch[\'input_ids\'].shape[-1])\n        call_file=folder/f\'{index:03d}.json\'\n        call_file.write_text(json.dumps(dict(prompt=text,images=captured,status=\'started\',prompt_tokens=length),ensure_ascii=False))\n        if length>self.config.max_input_tokens:raise ValueError(\'APC intermediate prompt exceeds max_input_tokens; no silent truncation.\')\n        inputs={k:v.to(device=self.device,dtype=self.dtype) if k in (\'pixel_values\',\'pixel_values_videos\') else v.to(self.device) for k,v in batch.items()}\n        last_text=\'\\n\'.join(c.get(\'text\',\'\') for c in messages[-1].get(\'content\',[]) if c.get(\'type\')==\'text\')\n        is_final=last_text.lstrip().startswith(\'Here is the question and the response you provided:\')\n        cap=min(max_new_tokens or self.config.max_new_tokens,self.config.final_max_new_tokens if is_final else self.config.max_new_tokens)\n        tokenizer=self.processor.tokenizer;pad=tokenizer.pad_token_id\n        if pad is None:pad=tokenizer.eos_token_id\n        with torch.inference_mode():ids=self.vlm_model.generate(**inputs,do_sample=False,num_beams=1,max_new_tokens=cap,use_cache=True,pad_token_id=pad)\n        new=ids[:,length:];raw=self.processor.batch_decode(new,skip_special_tokens=True,clean_up_tokenization_spaces=False)[0].strip()\n        call=dict(prompt_tokens=length,generated_tokens=int(new.shape[-1]),seconds=time.monotonic()-started,raw_output=raw[:240])\n        call.update(images=captured,status=\'complete\')\n        call_file.write_text(json.dumps(dict(call,prompt=text),ensure_ascii=False))\n        self.calls.append(call)\n        if self.callback:self.callback(\'vlm_return\',call=len(self.calls),vlm_seconds=call[\'seconds\'])\n        return raw\n',encoding='utf-8')
(SCRIPTS/'apc_experiment.py').write_text('"""One APC pipeline spanning two GPUs; baseline reuses exactly the same Qwen weights."""\nimport contextlib,copy,hashlib,json,logging,os,random,shutil,signal,sys,time,traceback\nfrom pathlib import Path\nimport numpy as np\nimport torch\nfrom PIL import Image\nfrom box import Box\nfrom apc_common import atomic_json,parse_option,read_jsonl\n\nclass APCExperiment:\n    def __init__(self,cfg,assets,callback=None,sanity=False):\n        self.cfg=cfg;self.sanity=sanity;self.callback=callback or (lambda *a,**k:None)\n        import yaml\n        self.callback(\'loading_qwen\')\n        config=Box(yaml.safe_load(Path(\'apc/configs/qwenvl2_5_7b_instruct.yaml\').read_text()))\n        root=Path(assets[\'root\'])\n        config.vlm.pretrained_model=str(root/\'qwen_model\');config.vlm.dtype=cfg[\'dtype\']\n        config.vlm.min_pixels=cfg[\'min_pixels\'];config.vlm.max_pixels=cfg[\'max_pixels\']\n        config.vlm.memory_report_dir=cfg[\'work\']\n        config.vlm.activation_reserve_gib=cfg[\'vlm_activation_reserve_gib\']\n        config.vlm.vision_reserve_gib=cfg[\'vision_reserve_gib\']\n        config.vlm.final_max_new_tokens=cfg[\'final_max_new_tokens\']\n        config.vlm.max_new_tokens=cfg[\'intermediate_max_new_tokens\'];config.vlm.max_input_tokens=cfg[\'max_input_tokens\']\n        config.detection.ckpt_path=str(root/\'checkpoints/groundingdino_swint_ogc.pth\')\n        config.detection.box_threshold=cfg[\'detection_box_threshold\'];config.detection.text_threshold=cfg[\'detection_text_threshold\']\n        config.segmentation.ckpt_path=str(root/\'checkpoints/sam_vit_h_4b8939.pth\');config.depth.ckpt_path=str(root/\'checkpoints/depth_pro.pt\')\n        config.orientation.ckpt_path=str(root/\'hf_cache\');config.orientation.model_path=str(root/\'checkpoints/orient_dino_weight.pt\')\n        config.orientation.processor_path=str(root/\'dinov2_large\')\n        # Replace only the VLM class, retaining APC\'s stages and prompts.\n        from apc import apc_pipeline\n        from apc_qwen_adapter import KaggleQwen\n        apc_pipeline.get_vlm_model_class=lambda kind:KaggleQwen if kind==\'qwenvl2_5\' else (_ for _ in ()).throw(ValueError(kind))\n        self.callback(\'loading_apc_pipeline\')\n        vision_initial_device=\'cpu\' if cfg[\'vision_offload\'] else \'cuda:1\'\n        self.apc=apc_pipeline.APC(config,device_vlm=\'cuda:0\',device_vision=vision_initial_device)\n        self.vlm=self.apc.vlm_model;self.vlm.callback=self.callback\n        if cfg[\'vision_offload\']:\n            from apc_kaggle_memory import install_vision_scheduler\n            self.vision_scheduler=install_vision_scheduler(self.apc,self.vlm,\'cuda:1\')\n        memory={str(i):dict(free_gib=torch.cuda.mem_get_info(i)[0]/2**30, allocated_gib=torch.cuda.memory_allocated(i)/2**30) for i in (0,1)}\n        atomic_json(Path(cfg[\'work\'])/\'pipeline_memory.json\',memory)\n        print(\'APC loaded, GPU memory:\',json.dumps(memory),flush=True)\n        # Use the repository option-selection prompt.\n        self.apc.logger.setLevel(logging.INFO if sanity else logging.WARNING)\n        # Compute depth only once per sample even when several objects are abstracted.\n        original_depth=self.apc.depth_module.run_depth_estimation\n        self.depth_cache=None\n        def depth_once(image):\n            if self.depth_cache is None:\n                self.depth_cache=original_depth(image)\n                trace=Path(os.environ[\'APC_TRACE_DIR\'])\n                np.save(trace/\'depth.npy\',self.depth_cache)\n                finite=self.depth_cache[np.isfinite(self.depth_cache)]\n                low,high=np.percentile(finite,[2,98]) if finite.size else (0,1)\n                preview=np.clip((np.nan_to_num(self.depth_cache)-low)/max(high-low,1e-6),0,1)\n                Image.fromarray((preview*255).astype(\'uint8\')).save(trace/\'depth_preview.png\')\n                (trace/\'depth_metadata.json\').write_text(json.dumps(dict(shape=list(self.depth_cache.shape),preview_percentiles=[float(low),float(high)])))\n            return self.depth_cache\n        self.apc.depth_module.run_depth_estimation=depth_once\n        # SAM embedding is also identical for every object of one input image.\n        predictor=self.apc.detection_module.segmentation_predictor;original_set=predictor.set_image;self.sam_image_hash=None\n        def set_once(image,*a,**kw):\n            digest=hashlib.sha256(np.ascontiguousarray(image).tobytes()).hexdigest()\n            if digest!=self.sam_image_hash:\n                result=original_set(image,*a,**kw);self.sam_image_hash=digest;return result\n        predictor.set_image=set_once\n        for name in (\'get_objects_of_interest\',\'do_scene_abstraction\',\'get_reference_viewer\',\'do_perspective_change\',\'do_perspective_prompting_visual\'):\n            original=getattr(self.apc,name)\n            def timed(*a,_name=name,_fn=original,**kw):\n                self.callback(_name)\n                if _name==\'do_perspective_prompting_visual\' and len(a)>1:\n                    a=list(a);a[1]=copy.deepcopy(a[1]);a=tuple(a)\n                result=_fn(*a,**kw)\n                if _name==\'get_objects_of_interest\' and (not isinstance(result,tuple) or not result[0]):raise ValueError(\'APC could not parse object names\')\n                if _name==\'do_scene_abstraction\':\n                    if len(result)<2:raise ValueError(\'APC detected no usable objects\')\n                    for obj in result.values():\n                        if not np.isfinite(obj[\'position\']).all() or not np.isfinite(obj[\'orientation\']).all():raise FloatingPointError(\'Nonfinite scene geometry\')\n                trace=Path(os.environ[\'APC_TRACE_DIR\'])\n                (trace/f\'{_name}.json\').write_text(json.dumps(dict(result=result),default=lambda x:x.tolist() if isinstance(x,np.ndarray) else str(x),ensure_ascii=False))\n                return result\n            setattr(self.apc,name,timed)\n        original_render=self.apc.render_module.render_visual_prompt\n        def render_prompt(*args,**kw):\n            self.callback(\'render_visual_prompt\')\n            result=original_render(*args,**kw)\n            if sanity:result.save(Path(kw[\'trace_save_dir\'])/\'apc_visual_prompt.png\')\n            return result\n        self.apc.render_module.render_visual_prompt=render_prompt\n        if sanity:\n            def finite_hook(module,args,output):\n                logits=getattr(output,\'logits\',None)\n                if logits is not None and not torch.isfinite(logits[:,-1,:]).all().item():raise FloatingPointError(\'NaN/Inf in Qwen raw logits\')\n            self.vlm.vlm_model.register_forward_hook(finite_hook)\n        self.callback(\'ready\')\n\n    def run(self,row,mode,scratch):\n        self.depth_cache=None;self.sam_image_hash=None;self.vlm.calls=[]\n        options=\'\\n\'.join(f\'{k}. {v}\' for k,v in row[\'choices\'].items())\n        # Choices carry object descriptions for questions such as "Which object is higher?".\n        question=row[\'question\']+\'\\nAnswer options:\\n\'+options\n        with Image.open(row[\'image_path\']) as image:\n            image=image.convert(\'RGB\')\n            with torch.inference_mode():\n                if mode==\'direct\':raw=self.vlm.generate_response(question+\'\\nAnswer with the option letter only.\',image,max_new_tokens=self.cfg[\'final_max_new_tokens\'])\n                elif mode==\'apc_vis\':\n                    raw,_=self.apc.run_apc(image,question,trace_save_dir=str(scratch),perspective_prompt_type=\'visual\',\n                         visualize_trace=False,visualize_scene_abstraction=False,\n                         render_whole_scene=row[\'category\']==\'orientation_on_the_left\',return_conv_history=False,options=options)\n                else:raise ValueError(mode)\n        if not isinstance(raw,str):raise TypeError(\'APC response must be text\')\n        return raw\n\ndef is_fatal(exc):\n    text=str(exc).lower()\n    return isinstance(exc,torch.cuda.OutOfMemoryError) or isinstance(exc,RuntimeError) and any(s in text for s in (\'device-side assert\',\'illegal memory\',\'cuda error\',\'cublas_status\',\'out of memory\'))\n\ndef main():\n    import argparse\n    parser=argparse.ArgumentParser();parser.add_argument(\'--manifest\',required=True);parser.add_argument(\'--smoke\',action=\'store_true\');a=parser.parse_args()\n    m=json.loads(Path(a.manifest).read_text());cfg=m[\'config\'];out=Path(cfg[\'output\']);folder=out/(\'smoke\' if a.smoke else \'full\');folder.mkdir(exist_ok=True)\n    rows=[json.loads(x) for x in (out/\'samples.jsonl\').read_text().splitlines()]\n    if a.smoke:rows=rows[:1]\n    state_path=folder/\'state.json\';started=time.monotonic();state=dict(stage=\'starting\',processed=0,total=len(rows)*len(cfg[\'modes\']),new_processed=0,errors=0,invalid=0)\n    def update(stage,**kw):\n        state.update(stage=stage,updated_unix=time.time(),elapsed_seconds=time.monotonic()-started,**kw);atomic_json(state_path,state)\n    previous={mode:({} if a.smoke else read_jsonl(folder/f\'predictions_{mode}.jsonl\',repair=True)) for mode in cfg[\'modes\']}\n    allowed={r[\'sample_id\'] for r in rows}\n    for mode,values in previous.items():\n        for sid,r in values.items():assert sid in allowed and r[\'fingerprint\']==m[\'fingerprint\'] and r[\'mode\']==mode,\'Resume mismatch\'\n    pending=[]\n    for row in rows:\n        for mode in cfg[\'modes\']:\n            old=previous[mode].get(row[\'sample_id\'])\n            if not old or cfg[\'retry_failed\'] and old[\'status\']!=\'ok\':pending.append((row,mode))\n    state[\'processed\']=state[\'total\']-len(pending);update(\'initializing\')\n    if not pending:update(\'complete\');return\n    assets=json.loads(Path(cfg[\'work\'],\'assets_ready.json\').read_text())\n    try:\n        experiment=APCExperiment(cfg,assets,update,sanity=a.smoke)\n        handles={mode:(folder/f\'predictions_{mode}.jsonl\').open(\'a\',encoding=\'utf-8\') for mode in cfg[\'modes\']}\n        try:\n            for index,(row,mode) in enumerate(pending,1):\n                update(\'sample_start\',sample_id=row[\'sample_id\'],mode=mode,sample_started_unix=time.time())\n                seed=int.from_bytes(hashlib.sha256(f"{cfg[\'seed\']}:{row[\'sample_id\']}".encode()).digest()[:4],\'big\')\n                random.seed(seed);np.random.seed(seed);torch.manual_seed(seed);torch.cuda.manual_seed_all(seed)\n                for gpu in (0,1):torch.cuda.reset_peak_memory_stats(gpu)\n                scratch=Path(cfg[\'work\'])/\'scratch\'/hashlib.sha256((mode+row[\'sample_id\']).encode()).hexdigest()[:16];scratch.mkdir(parents=True,exist_ok=True)\n                record=dict(sample_id=row[\'sample_id\'],fingerprint=m[\'fingerprint\'],mode=mode,status=\'error\',prediction=None,raw_output=\'\',seed=seed)\n                fatal=None;t=time.monotonic()\n                def timeout(signum,frame):raise TimeoutError(\'APC sample exceeded timeout\')\n                old_handler=signal.signal(signal.SIGALRM,timeout);signal.alarm(cfg[\'sample_timeout_seconds\'])\n                try:\n                    raw=experiment.run(row,mode,scratch);prediction=parse_option(raw,row[\'choices\'])\n                    record.update(status=\'ok\',raw_output=raw,prediction=prediction,parse_valid=prediction is not None,\n                         output_issue=\'empty_output\' if not raw else \'degenerate_exclamation\' if re.fullmatch(r\'!{4,}\',raw) else \'unparsed_answer\' if prediction is None else None)\n                    if prediction is None:state[\'invalid\']+=1\n                    if a.smoke and prediction is None:fatal=ValueError(\'Smoke response was not an exact answer option\')\n                except Exception as exc:\n                    state[\'errors\']+=1;record.update(error_type=type(exc).__name__,message=str(exc));traceback.print_exc()\n                    if a.smoke or is_fatal(exc):fatal=exc\n                finally:\n                    signal.alarm(0);signal.signal(signal.SIGALRM,old_handler)\n                    record.update(seconds=time.monotonic()-t,vlm_calls=experiment.vlm.calls,\n                       peak_vram_gib={str(i):torch.cuda.max_memory_allocated(i)/2**30 for i in (0,1)})\n                    f=handles[mode];f.write(json.dumps(record,ensure_ascii=False)+\'\\n\');f.flush()\n                    if index%8==0 or fatal is not None or index==len(pending):os.fsync(f.fileno())\n                    if a.smoke:\n                        destination=folder/f\'trace_{mode}\'\n                        if not destination.exists():shutil.copytree(scratch,destination)\n                    resolved=scratch.resolve();resolved.relative_to((Path(cfg[\'work\'])/\'scratch\').resolve());shutil.rmtree(resolved)\n                state[\'processed\']+=1;state[\'new_processed\']+=1\n                update(\'sample_done\',last_output=record[\'raw_output\'][:180],last_seconds=record[\'seconds\'],sample_started_unix=None)\n                if index<=3 or index%10==0:print(json.dumps(state),flush=True)\n                if fatal:raise fatal\n        finally:\n            for f in handles.values():f.close()\n        update(\'complete\')\n    except BaseException as exc:update(\'error\',error_type=type(exc).__name__,message=str(exc));raise\n\nif __name__==\'__main__\':\n    import re\n    main()\n',encoding='utf-8')
(SCRIPTS/'apc_score.py').write_text('"""Score both paired modes with a full denominator; never drop failed APC samples."""\nimport collections,json,statistics,sys\nfrom pathlib import Path\nfrom apc_common import read_jsonl\nout=Path(sys.argv[1]);m=json.loads((out/\'manifest.json\').read_text());samples=[json.loads(x) for x in (out/\'samples.jsonl\').read_text().splitlines()]\ntruth={r[\'sample_id\']:r for r in samples};groups=collections.defaultdict(list)\nfor r in samples:groups[r[\'circular_group\']].append(r)\nmetrics={}\nfor mode in m[\'config\'][\'modes\']:\n records=read_jsonl(out/\'full\'/f\'predictions_{mode}.jsonl\')\n assert set(records).issubset(truth)\n for r in records.values():assert r[\'fingerprint\']==m[\'fingerprint\']\n def hit(sid):\n  p=records.get(sid,{});return p.get(\'status\')==\'ok\' and p.get(\'prediction\')==truth[sid][\'answer\']\n merged=[{**t,**records.get(sid,dict(status=\'missing\',prediction=None)),\'correct\':bool(hit(sid))} for sid,t in truth.items()]\n grouped=[dict(circular_group=k,category=v[0][\'category\'],paper_task=v[0].get(\'paper_task\',v[0][\'category\']),correct=all(hit(r[\'sample_id\']) for r in v)) for k,v in groups.items()]\n successful=[r for r in records.values() if r[\'status\']==\'ok\'];invalid=[r for r in successful if r.get(\'prediction\') is None]\n mean=lambda xs:statistics.mean(xs) if xs else None\n metric=dict(expected_rows=len(samples),attempted_rows=len(records),successful_rows=len(successful),missing_rows=len(samples)-len(records),\n  pipeline_errors=sum(r[\'status\']!=\'ok\' for r in records.values()),invalid_outputs=len(invalid),\n  invalid_by_reason=dict(collections.Counter(r.get(\'output_issue\',\'unparsed_answer\') for r in invalid)),\n  row_accuracy=mean([r[\'correct\'] for r in merged]),circular_groups=len(grouped),circular_accuracy=mean([r[\'correct\'] for r in grouped]),\n  circular_by_subtype={k:mean([r[\'correct\'] for r in grouped if r[\'category\']==k]) for k in sorted({r[\'category\'] for r in grouped})},\n  circular_by_paper_task={k:mean([r[\'correct\'] for r in grouped if r[\'paper_task\']==k]) for k in sorted({r[\'paper_task\'] for r in grouped})},\n  selected_subtasks=sorted({r[\'category\'] for r in samples}),scope=m[\'config\'].get(\'scope\'),\n  coverage=len(successful)/len(samples),all_rows_attempted=len(records)==len(samples),complete=len(successful)==len(samples),\n  mean_seconds=mean([r[\'seconds\'] for r in successful]),parse_policy=m[\'config\'][\'parse_policy\'],\n  protocol=\'Paper task families with visibility recast from front/behind; Qwen2.5-VL-7B direct/APC-Vis, no LLM judge; not exact reproduction of paper Table 1\')\n for name,values in [(\'predictions\',merged),(\'circular_groups\',grouped),(\'invalid_outputs\',[r for r in merged if r[\'status\']==\'ok\' and r.get(\'prediction\') is None]),(\'pipeline_errors\',[r for r in merged if r[\'status\']!=\'ok\'])]:\n  (out/(mode+\'_\'+name+\'.jsonl\')).write_text(\'\'.join(json.dumps(r,ensure_ascii=False)+\'\\n\' for r in values),encoding=\'utf-8\')\n metrics[mode]=metric\n(out/\'summary.json\').write_text(json.dumps(metrics,indent=2),encoding=\'utf-8\');print(json.dumps(metrics,indent=2))\n\nif set((\'direct\',\'apc_vis\')).issubset(metrics):\n    a=read_jsonl(out/\'full/predictions_direct.jsonl\');b=read_jsonl(out/\'full/predictions_apc_vis.jsonl\')\n    paired=collections.Counter()\n    for sid,t in truth.items():\n        ha=a.get(sid,{}).get(\'status\')==\'ok\' and a.get(sid,{}).get(\'prediction\')==t[\'answer\']\n        hb=b.get(sid,{}).get(\'status\')==\'ok\' and b.get(sid,{}).get(\'prediction\')==t[\'answer\']\n        paired[\'both_correct\' if ha and hb else \'direct_only\' if ha else \'apc_only\' if hb else \'both_wrong\']+=1\n    comparison=dict(paired_rows=dict(paired),row_accuracy_delta_pp=100*(metrics[\'apc_vis\'][\'row_accuracy\']-metrics[\'direct\'][\'row_accuracy\']),\n        circular_accuracy_delta_pp=100*(metrics[\'apc_vis\'][\'circular_accuracy\']-metrics[\'direct\'][\'circular_accuracy\']),\n        circular_delta_by_paper_task_pp={k:100*(metrics[\'apc_vis\'][\'circular_by_paper_task\'][k]-metrics[\'direct\'][\'circular_by_paper_task\'][k]) for k in metrics[\'direct\'][\'circular_by_paper_task\']})\n    (out/\'paired_comparison.json\').write_text(json.dumps(comparison,indent=2),encoding=\'utf-8\')\n',encoding='utf-8')
(SCRIPTS/'apc_prepare_data.py').write_text("import base64,csv,hashlib,io,json,os,re,sys,shutil\nfrom pathlib import Path\nfrom PIL import Image\nfrom huggingface_hub import HfApi,hf_hub_download\nfrom apc_common import atomic_json,choose_samples\ncfg=json.loads(Path(sys.argv[1]).read_text());work=Path(cfg['work']);out=Path(cfg['output']);out.mkdir(parents=True,exist_ok=True)\nassets=json.loads((work/'assets_ready.json').read_text())\nif cfg['sanity']:\n    source=Path(cfg['sanity_image']) if cfg['sanity_image'] else Path(cfg['repo'])/'demo/sample_image_man.jpg'\n    with Image.open(source) as im:im.verify()\n    items=[dict(sample_id='sanity',question_id='repo_demo',question=cfg['sanity_question'],choices=cfg['sanity_choices'],answer=None,category='orientation_on_the_left',paper_task='left_right',circular_group='repo_demo',image_path=str(source))]\n    data_sha=None;all_count=1;columns=[]\nelse:\n    old=json.loads((out/'manifest.json').read_text()) if (out/'manifest.json').exists() else None\n    revision=old['data_sha'] if old else cfg['data_revision']\n    if cfg.get('dataset_input'):\n        tsv=Path(cfg['dataset_input']);data_sha='local:'+hashlib.sha256(tsv.read_bytes()).hexdigest()\n    else:\n        api=HfApi();info=api.dataset_info(cfg['dataset_id'],revision=revision,files_metadata=True);data_sha=info.sha\n        entry=next(f for f in info.siblings if f.rfilename==cfg['data_file'])\n        if shutil.disk_usage(work).free<(entry.size or 0)+1.5*2**30:raise RuntimeError('Need TSV size + 1.5 GiB for images; attach dataset_input or free workspace.')\n        tsv=hf_hub_download(cfg['dataset_id'],cfg['data_file'],repo_type='dataset',revision=data_sha)\n    images_dir = work / 'data/images'\n    images_dir.mkdir(parents=True, exist_ok=True)\n    csv.field_size_limit(2**31 - 1)\n    items, refs, unresolved = [], {}, []\n\n    def missing(x):\n        return x is None or str(x).strip().lower() in ('', 'nan', 'none', 'null')\n\n    with open(tsv, encoding='utf-8-sig', newline='') as f:\n        reader = csv.DictReader(f, delimiter='\\t')\n        columns = reader.fieldnames\n        required = {'index', 'qid', 'question', 'image', 'answer', 'category'}\n        if not required.issubset(columns or []):\n            raise ValueError(f'TSV schema changed: {columns}')\n        print('TSV columns:', columns, flush=True)\n        for rownum, row in enumerate(reader):\n            sid = str(row['index'])\n            qid = str(row['qid'])\n            value = str(row['image']).strip()\n            choices = {k: row[k] for k in 'ABCD' if not missing(row.get(k))}\n            answer = row['answer'].strip().upper()\n            if len(choices) < 2 or answer not in choices:\n                raise ValueError(f'Invalid choices/answer: {sid}')\n            item = dict(sample_id=sid, question_id=qid, question=row['question'],\n                        choices=choices, answer=answer, category=row['category'],\n                        circular_group=re.sub(r'-[0-3]$', '', qid), image_path=None)\n            # VLMEvalKit may store an index reference instead of repeating base64.\n            if value in refs:\n                item['image_path'] = refs[value]\n            elif re.fullmatch(r'\\d+', value or ''):\n                unresolved.append((item, value))\n            else:\n                payload = value.split(',', 1)[1] if value.startswith('data:image/') else value\n                raw = base64.b64decode(payload, validate=True)\n                target = images_dir / (hashlib.sha256(raw).hexdigest() + '.img')\n                if not target.is_file() or target.stat().st_size != len(raw):\n                    with Image.open(io.BytesIO(raw)) as image:\n                        image.verify()\n                    tmp = target.with_suffix('.tmp')\n                    tmp.write_bytes(raw)\n                    tmp.replace(target)\n                item['image_path'] = str(target)\n            if item['image_path']:\n                refs[sid] = item['image_path']\n            items.append(item)\n            if (rownum + 1) % 1000 == 0:\n                print('Prepared rows:', rownum + 1, flush=True)\n\n    while unresolved:\n        pending = []\n        for item, key in unresolved:\n            if key in refs:\n                item['image_path'] = refs[key]\n                refs[item['sample_id']] = refs[key]\n            else:\n                pending.append((item, key))\n        if len(pending) == len(unresolved):\n            raise ValueError(f'Unresolved image references: {[k for _, k in pending[:5]]}')\n        unresolved = pending\n    assert len({r['sample_id'] for r in items}) == len(items), 'Duplicate sample IDs'\n    assert all(Path(r['image_path']).is_file() for r in items)\n\n    all_count=len(items);items=choose_samples(items,cfg)\n    if not items:raise ValueError('No selected rows')\nmanifest=dict(config=cfg,model_sha=assets['snapshots']['qwen_model']['sha'],data_sha=data_sha,\n    selected_rows=len(items),total_dataset_rows=all_count,circular_groups=len({r['circular_group'] for r in items}),\n    source_lock=json.loads((work/'source_lock.json').read_text()),patches=json.loads((work/'patches.json').read_text()),assets=assets,\n    protocol='Paper task families: left/right, visibility recast from front/behind, facing; 7B direct vs APC-Vis; exact-choice without LLM judge; FP16 Kaggle runtime; exact-choice scoring')\ntext=''.join(json.dumps(r,ensure_ascii=False)+'\\n' for r in items)\nmanifest['fingerprint']=hashlib.sha256((json.dumps(manifest,sort_keys=True)+text).encode()).hexdigest()\nold_path=out/'manifest.json'\nif old_path.exists() and json.loads(old_path.read_text())['fingerprint']!=manifest['fingerprint']:\n    raise RuntimeError('Existing experiment has another fingerprint; choose a new experiment name, never rewrite old predictions.')\n(out/'samples.jsonl').write_text(text,encoding='utf-8');atomic_json(old_path,manifest)\nprint('Prepared',len(items),'rows /',manifest['circular_groups'],'groups; model SHA',manifest['model_sha'],flush=True)\n",encoding='utf-8')
(SCRIPTS/'apc_kaggle_memory.py').write_text('"""Plan whole decoder layers for APC + FP16 Qwen on two Kaggle GPUs."""\n\nGIB = 2**30\n\n\ndef validate_partition_map(device_map, tensor_names=None):\n    """Reject mappings that make Accelerate recursively move a parent and its children."""\n    if not device_map or "" in device_map:\n        raise ValueError("A partitioned Qwen map must not contain the root key \'\'.")\n    names = list(device_map)\n    if any(device not in (0, 1) for device in device_map.values()):\n        raise ValueError("The Qwen partition must use only GPU 0 and GPU 1.")\n    for parent in names:\n        for child in names:\n            if child.startswith(parent + "."):\n                raise ValueError(f"Overlapping device-map entries: {parent!r} and {child!r}.")\n    if tensor_names is not None:\n        for name in tensor_names:\n            matches = [prefix for prefix in names if name == prefix or name.startswith(prefix + ".")]\n            if len(matches) != 1:\n                raise ValueError(f"Tensor {name!r} must have exactly one explicit device-map owner.")\n\n\ndef verify_loaded_partition(model, device_map):\n    """Check real tensor devices after dispatch, including nonpersistent buffers."""\n    tensors = list(model.named_parameters()) + list(model.named_buffers())\n    validate_partition_map(device_map, [name for name, _ in tensors])\n    byte_counts = [0, 0]\n    for name, tensor in tensors:\n        prefix = next(prefix for prefix in device_map if name == prefix or name.startswith(prefix + "."))\n        expected = device_map[prefix]\n        if tensor.device.type != "cuda" or tensor.device.index != expected:\n            raise RuntimeError(f"Tensor {name} is on {tensor.device}; expected cuda:{expected}.")\n        byte_counts[expected] += tensor.numel() * tensor.element_size()\n    return {"verified": True, "tensor_count": len(tensors),\n            "tensor_gib_by_gpu": [value / GIB for value in byte_counts]}\n\n\ndef plan_qwen_device_map(module_sizes, num_layers, free_bytes, *,\n                         text_prefix="model.language_model", visual_prefix="model.visual",\n                         activation_reserve_gib=(3.0, 2.0), vision_reserve_gib=6.0,\n                         tie_word_embeddings=False):\n    """Return an explicit GPU-only map, reserving GPU 1 for the APC vision stack.\n\n    Unlike an automatic balanced map, this does not equalize Qwen weights across\n    unequal GPU budgets. Decoder layers remain contiguous and never split.\n    module_sizes comes from accelerate.utils.modeling.compute_module_sizes.\n    """\n    if num_layers < 2 or len(free_bytes) != 2 or len(activation_reserve_gib) != 2:\n        raise ValueError("Expected at least two decoder layers and exactly two GPUs.")\n    reserves = [float(value) * GIB for value in activation_reserve_gib]\n    if min(reserves) < 0 or vision_reserve_gib < 0:\n        raise ValueError("Memory reserves cannot be negative.")\n    budgets = [int(free_bytes[0] - reserves[0]),\n               int(free_bytes[1] - reserves[1] - vision_reserve_gib * GIB)]\n    if min(budgets) <= 0:\n        raise RuntimeError("Insufficient free GPU memory after reserving APC/activation space. "\n                           "Stop old GPU workers or restart the Kaggle session.")\n    layers = [f"{text_prefix}.layers.{index}" for index in range(num_layers)]\n    embedding, norm = f"{text_prefix}.embed_tokens", f"{text_prefix}.norm"\n    for name in ("", visual_prefix, embedding, norm, *layers):\n        if name not in module_sizes:\n            raise ValueError(f"Qwen architecture/size table is missing {name!r}.")\n    if not tie_word_embeddings and "lm_head" not in module_sizes:\n        raise ValueError("Untied Qwen output head is missing from the size table.")\n    sizes = [int(module_sizes[name]) for name in layers]\n    head_size = int(module_sizes.get("lm_head", 0))\n    fixed_zero = int(module_sizes[visual_prefix] + module_sizes[embedding])\n    fixed_one = int(module_sizes[norm])\n    if tie_word_embeddings:\n        fixed_zero += head_size\n    else:\n        fixed_one += head_size\n    # The remaining small buffers belong to the explicitly mapped rotary module.\n    remainder = int(module_sizes[""]) - fixed_zero - fixed_one - sum(sizes)\n    if remainder < 0:\n        raise ValueError("Overlapping/inconsistent module sizes in the Qwen memory plan.")\n    fixed_zero += remainder\n    candidates = []\n    for cut in range(1, num_layers):\n        required = [fixed_zero + sum(sizes[:cut]), fixed_one + sum(sizes[cut:])]\n        if all(required[gpu] <= budgets[gpu] for gpu in (0, 1)):\n            candidates.append((cut, required))\n    if not candidates:\n        raise RuntimeError(\n            "FP16 Qwen + APC cannot fit the configured GPU reserves. "\n            f"Qwen={module_sizes[\'\']/GIB:.2f} GiB; weight budgets="\n            f"{budgets[0]/GIB:.2f}/{budgets[1]/GIB:.2f} GiB. "\n            "No CPU/disk offload or quantization was applied. Inspect memory_plan.json."\n        )\n    # Keep as much Qwen on GPU 0 as fits; GPU 1 needs space for vision inference.\n    cut, required = max(candidates, key=lambda candidate: candidate[0])\n    device_map = {visual_prefix: 0, embedding: 0,\n                  f"{text_prefix}.rotary_emb": 0, norm: 1,\n                  "lm_head": 0 if tie_word_embeddings else 1}\n    device_map.update({name: 0 if index < cut else 1 for index, name in enumerate(layers)})\n    validate_partition_map(device_map)\n    report = {\n        "strategy": "contiguous_nonoverlapping_decoder_layers_v2", "device_map": device_map,\n        "layers_gpu0": cut, "layers_gpu1": num_layers - cut,\n        "qwen_weights_gib": module_sizes[""] / GIB,\n        "free_before_qwen_gib": [value / GIB for value in free_bytes],\n        "weight_budget_gib": [value / GIB for value in budgets],\n        "planned_weights_gib": [value / GIB for value in required],\n        "activation_reserve_gib": list(activation_reserve_gib),\n        "vision_reserve_gpu1_gib": vision_reserve_gib,\n        "quantized": False, "cpu_or_disk_offload": False,\n    }\n    return device_map, report\n\n\ndef make_qwen_memory_plan(model_dir, dtype, *, activation_reserve_gib=(3.0, 2.0),\n                          vision_reserve_gib=6.0):\n    """Inspect the model on the meta device; allocate no real model weights."""\n    import gc\n    import torch\n    from accelerate import init_empty_weights\n    from accelerate.utils.modeling import check_device_map, compute_module_sizes\n    from transformers import AutoConfig, Qwen2_5_VLForConditionalGeneration\n\n    if torch.cuda.device_count() != 2:\n        raise RuntimeError("Select Kaggle GPU T4 x2 before preparing the Qwen device map.")\n    config = AutoConfig.from_pretrained(model_dir, local_files_only=True)\n    config._attn_implementation = "sdpa"\n    with init_empty_weights(include_buffers=True):\n        skeleton = Qwen2_5_VLForConditionalGeneration(config)\n        skeleton.tie_weights()\n    if hasattr(skeleton.model, "language_model"):\n        text_prefix, visual_prefix = "model.language_model", "model.visual"\n    else:\n        text_prefix, visual_prefix = "model", "visual"\n    text_model = skeleton.get_submodule(text_prefix)\n    # Initialize each CUDA context before measuring usable memory.\n    for gpu in (0, 1):\n        with torch.cuda.device(gpu):\n            torch.empty(1, device=f"cuda:{gpu}")\n            torch.cuda.empty_cache()\n    free_bytes = [torch.cuda.mem_get_info(gpu)[0] for gpu in (0, 1)]\n    device_map, report = plan_qwen_device_map(\n        compute_module_sizes(skeleton, dtype=dtype), len(text_model.layers), free_bytes,\n        text_prefix=text_prefix, visual_prefix=visual_prefix,\n        activation_reserve_gib=activation_reserve_gib, vision_reserve_gib=vision_reserve_gib,\n        tie_word_embeddings=getattr(config.get_text_config(), "tie_word_embeddings", False),\n    )\n    validate_partition_map(device_map, [name for name, _ in skeleton.named_parameters()]\n                           + [name for name, _ in skeleton.named_buffers()])\n    check_device_map(skeleton, device_map)\n    del text_model, skeleton\n    gc.collect()\n    return device_map, report\n\n\nclass VisionScheduler:\n    """Keep one APC vision model on GPU 1 at a time, parking idle weights in RAM."""\n\n    def __init__(self, models, predictor, device="cuda:1", torch_module=None):\n        if torch_module is None:\n            import torch as torch_module\n        self.torch = torch_module\n        self.models = models\n        self.predictor = predictor\n        self.device = device\n        self.active = None\n\n    def park_all(self):\n        if self.active is None:\n            return\n        with self.torch.cuda.device(self.device):\n            if self.active == "sam" and self.torch.is_tensor(self.predictor.features):\n                self.predictor.features = self.predictor.features.to("cpu")\n            self.models[self.active].to("cpu")\n            self.active = None\n            self.torch.cuda.empty_cache()\n\n    def activate(self, name):\n        if name == self.active:\n            return\n        self.park_all()\n        with self.torch.cuda.device(self.device):\n            self.models[name].to(self.device)\n            if name == "sam" and self.torch.is_tensor(self.predictor.features):\n                self.predictor.features = self.predictor.features.to(self.device)\n        self.active = name\n\n\ndef install_vision_scheduler(apc, vlm, device="cuda:1"):\n    """Install scheduling after initializing the auxiliary models on CPU.\n\n    Model precision, inference, prompts, and thresholds are preserved. Moving the\n    SAM cached embedding with its model allows VLM refinement calls between masks.\n    """\n    from functools import wraps\n    from torchvision.transforms import Lambda\n\n    detection, depth, orientation = apc.detection_module, apc.depth_module, apc.orientation_module\n    predictor = detection.segmentation_predictor\n    models = {"grounding": detection.detection_model, "sam": detection.segmentation_model,\n              "depth": depth.depth_model, "orientation": orientation.orientation_model}\n    scheduler = VisionScheduler(models, predictor, device)\n    for module in (detection, depth, orientation):\n        module.device = device\n    apc.device_vision = device\n    apc.render_module.device = device\n    # Depth Pro closes over its initialization device in the preprocessing Lambda.\n    # Retain the original GPU normalization/dtype order when weights start on CPU.\n    transform = depth.depth_transform\n    if not hasattr(transform, "transforms") or len(transform.transforms) < 4:\n        raise RuntimeError("Unexpected Depth Pro preprocessing; inspect its upstream revision.")\n    transform.transforms[1] = Lambda(lambda tensor: tensor.to(device))\n\n    def wrap(owner, method_name, model_name):\n        method = getattr(owner, method_name)\n\n        @wraps(method)\n        def scheduled(*args, **kwargs):\n            scheduler.activate(model_name)\n            return method(*args, **kwargs)\n\n        setattr(owner, method_name, scheduled)\n\n    wrap(detection, "run_detection", "grounding")\n    for method in ("set_image", "predict", "predict_torch"):\n        wrap(predictor, method, "sam")\n    wrap(depth, "run_depth_estimation", "depth")\n    wrap(orientation, "get_3angle", "orientation")\n    vlm.before_generate = scheduler.park_all\n    return scheduler\n',encoding='utf-8')
(SCRIPTS/'apc_qwen_preflight.py').write_text('"""Check the real Accelerate dispatch path before compiling APC\'s native dependencies."""\n\nimport json\nimport sys\nfrom pathlib import Path\n\n\ndef probe_dispatch(device_map):\n    import gc\n    import torch\n    from accelerate import dispatch_model\n    from apc_kaggle_memory import validate_partition_map, verify_loaded_partition\n\n    validate_partition_map(device_map)\n\n    class Probe(torch.nn.Module):\n        def __init__(self):\n            super().__init__()\n            self.order = list(device_map)\n            for name in self.order:\n                parent = self\n                parts = name.split(".")\n                for part in parts[:-1]:\n                    if not hasattr(parent, part):\n                        parent.add_module(part, torch.nn.Module())\n                    parent = getattr(parent, part)\n                leaf = torch.nn.Linear(4, 4)\n                with torch.no_grad():\n                    leaf.weight.copy_(torch.eye(4))\n                    leaf.bias.zero_()\n                leaf.register_buffer("probe_buffer", torch.ones(1), persistent=False)\n                parent.add_module(parts[-1], leaf)\n\n        def forward(self, value):\n            for name in self.order:\n                value = self.get_submodule(name)(value)\n            return value\n\n    # The same module paths/map as Qwen, but only a few KiB of actual weights.\n    model = dispatch_model(Probe().eval(), device_map=device_map)\n    result = verify_loaded_partition(model, device_map)\n    with torch.inference_mode():\n        value = torch.ones((1, 4), device="cuda:0")\n        output = model(value)\n        if not torch.allclose(output, value.to(output.device)):\n            raise RuntimeError("Accelerate cross-GPU dispatch probe returned an incorrect value.")\n    result.update(cross_gpu_forward=True, model="tiny_identity_probe", gpu_names=[\n        torch.cuda.get_device_name(index) for index in (0, 1)])\n    del model, value, output\n    gc.collect()\n    for index in (0, 1):\n        with torch.cuda.device(index):\n            torch.cuda.empty_cache()\n    return result\n\n\ndef main(config_path):\n    import torch\n    from huggingface_hub import hf_hub_download\n    from apc_kaggle_memory import make_qwen_memory_plan\n\n    config = json.loads(Path(config_path).read_text())\n    work = Path(config["work"])\n    model_dir = Path(config["asset_input"] or config["asset_root"]) / "qwen_model"\n    if not (model_dir / "config.json").is_file():\n        if config["asset_input"]:\n            raise RuntimeError("The mounted asset_input is missing qwen_model/config.json.")\n        hf_hub_download(config["model_id"], "config.json", revision=config["model_revision"],\n                        local_dir=str(model_dir))\n    device_map, report = make_qwen_memory_plan(\n        str(model_dir), getattr(torch, config["dtype"]),\n        activation_reserve_gib=config["vlm_activation_reserve_gib"],\n        vision_reserve_gib=config["vision_reserve_gib"],\n    )\n    report["dispatch_probe"] = probe_dispatch(device_map)\n    (work / "dispatch_preflight.json").write_text(json.dumps(report, indent=2))\n    print("PASS: Qwen map has no root/parent overlaps; real Accelerate dispatch and "\n          "a tiny forward succeeded on both GPUs.", flush=True)\n    print(json.dumps({key: value for key, value in report.items() if key != "device_map"}), flush=True)\n\n\nif __name__ == "__main__":\n    main(sys.argv[1])\n',encoding='utf-8')

## Cấu hình

Chỉ chạy mode `apc_vis`. Phạm vi của notebook là facing.


In [ ]:
CFG = {'model_id': 'Qwen/Qwen2.5-VL-7B-Instruct',
 'model_revision': 'cc594898137f460bfe9f0759e9844b3ce807cfb5',
 'dataset_id': 'ccvl/3DSRBench',
 'data_revision': '5c90e422b68d0484d50a3ff06b3096ea7d288026',
 'data_file': '3dsrbench_v1_vlmevalkit_circular.tsv',
 'dataset_input': None,
 'asset_input': None,
 'asset_root': '/kaggle/temp/apc_7b_assets',
 'experiment': 'apc_qwen7b_full_part2_facing_record_fix_v1',
 'scope': 'paper_tasks_visibility',
 'modes': ['apc_vis'],
 'seed': 42,
 'dtype': 'float16',
 'min_pixels': None,
 'max_pixels': None,
 'intermediate_max_new_tokens': 1024,
 'final_max_new_tokens': 512,
 'max_input_tokens': 32768,
 'detection_box_threshold': 0.05,
 'detection_text_threshold': 0.05,
 'limit': None,
 'sample_timeout_seconds': 900,
 'load_timeout_seconds': 1800,
 'retry_failed': False,
 'parse_policy': 'strict_letter_or_unique_exact_choice_v1',
 'source_snapshot_sha': '9059ed359baf89b49675ed3e0407f843b3ff28de78e5cae1c870adf79f44a339',
 'runtime_sha': '31499644d354fb713f1ce06e87fe300e2d715f61616180166731f1360cb4fa25',
 'sanity': False,
 'vlm_activation_reserve_gib': [3.0, 2.0],
 'vision_reserve_gib': 6.0,
 'vision_offload': True,
 'paper_tasks': ['facing'],
 'sanity_sample_id': None}
assert CFG['modes'] == ['apc_vis'], 'APC-Vis only'
assert CFG['dtype']=='float16', 'This T4 x2 configuration uses FP16.'
assert CFG['scope']=='paper_tasks_visibility', 'This notebook evaluates only the three paper task families.'
assert CFG['limit'] is None or isinstance(CFG['limit'],int) and CFG['limit']>0
assert Path(CFG['experiment']).name==CFG['experiment']
CFG.update(work=str(WORK),repo=str(REPO),output=str(WORK/('sanity/'+CFG['runtime_sha'][:12] if CFG['sanity'] else 'results/'+CFG['experiment'])))
CONFIG=WORK/('sanity_config.json' if CFG['sanity'] else 'full_config.json')
CONFIG.write_text(json.dumps(CFG,indent=2),encoding='utf-8');print(json.dumps(CFG,indent=2))

## Python/CUDA tách khỏi kernel

Dùng uv tránh ensurepip Python3.13; Python3.10, Torch2.4.1/cu124 và torchvision0.19.1; giữ đường dẫn driver NVIDIA, kiểm tra phép tính trên cả hai GPU.

In [ ]:
# Kiểm tra GPU trước khi tải/cài package. Không cần import torch của kernel.
import csv
smi = shutil.which('nvidia-smi')
if not smi:
    smi = next((p for p in ('/usr/bin/nvidia-smi', '/usr/local/nvidia/bin/nvidia-smi') if Path(p).is_file()), None)
if not smi:
    raise RuntimeError('Phiên Kaggle chưa có nvidia-smi. Chọn Settings > Accelerator > GPU T4 x2, rồi khởi động lại session. Cài package không thể cấp GPU cho phiên CPU.')
gpu_query = subprocess.run(
    [smi, '--query-gpu=uuid,name,driver_version,memory.total', '--format=csv,noheader,nounits'],
    capture_output=True, text=True)
print(gpu_query.stdout)
if gpu_query.returncode != 0:
    print(gpu_query.stderr)
    raise RuntimeError('NVIDIA driver chưa truy cập được. Chọn accelerator 2 GPU và khởi động lại session Kaggle.')
gpu_rows = [row for row in csv.reader(gpu_query.stdout.splitlines()) if len(row) == 4]
if len(gpu_rows) < 2:
    raise RuntimeError(f'Phiên này chỉ có {len(gpu_rows)} GPU. Full benchmark cần hai GPU; khởi động lại session sau khi chọn accelerator.')
GPU_IDS = [row[0].strip() for row in gpu_rows[:2]]
print('GPU UUID được dùng:', GPU_IDS)

minimum_env_space = (0.25 if (WORK / 'venv_py310_uv/pyvenv.cfg').is_file() else 8) * 2**30
if shutil.disk_usage(WORK).free < minimum_env_space:
    raise RuntimeError('Không đủ working cho cài môi trường (8 GiB khi cài mới, 0.25 GiB khi tiếp tục). Dùng model_input hoặc session mới nếu cần.')

# Bootstrap độc lập với pip/ensurepip/Python version của kernel Kaggle.
import io, platform, tarfile, urllib.request
assert platform.system() == 'Linux' and platform.machine() in ('x86_64', 'AMD64'), 'Cell này dành cho Kaggle Linux x64.'
UV_VERSION = '0.9.5'
UV_DIR = WORK / 'tools' / ('uv-' + UV_VERSION)
UV = UV_DIR / 'uv'
if not UV.exists():
    asset = 'uv-x86_64-unknown-linux-gnu.tar.gz'
    base_url = f'https://github.com/astral-sh/uv/releases/download/{UV_VERSION}/'
    with urllib.request.urlopen(base_url + asset, timeout=120) as response:
        raw = response.read()
    with urllib.request.urlopen(base_url + asset + '.sha256', timeout=120) as response:
        expected = response.read().decode().split()[0]
    assert hashlib.sha256(raw).hexdigest() == expected, 'uv checksum mismatch'
    UV_DIR.mkdir(parents=True, exist_ok=True)
    # Chỉ lấy đúng binary; không extract đường dẫn bất kỳ từ tar.
    with tarfile.open(fileobj=io.BytesIO(raw), mode='r:gz') as archive:
        members = [m for m in archive.getmembers() if Path(m.name).name == 'uv' and m.isfile()]
        assert len(members) == 1
        with archive.extractfile(members[0]) as binary:
            UV.write_bytes(binary.read())
    UV.chmod(0o755)

UV.chmod(UV.stat().st_mode | 0o111)

BOOT_ENV = os.environ.copy()
# Chỉ bỏ đường dẫn Python của kernel; GIỮ đường dẫn thư viện NVIDIA của Kaggle.
for key in ('PYTHONPATH', 'PYTHONHOME', 'VIRTUAL_ENV'):
    BOOT_ENV.pop(key, None)
BOOT_ENV['PYTHONNOUSERSITE'] = '1'
BOOT_ENV['UV_PYTHON_INSTALL_DIR'] = str(WORK / 'python_runtimes')
BOOT_ENV['UV_CACHE_DIR'] = str(WORK / 'uv_cache')
BOOT_ENV['UV_NO_PROGRESS'] = '1'
BOOT_ENV['UV_LINK_MODE'] = 'copy'
BOOT_ENV.pop('UV_SYSTEM_PYTHON', None)
# Dùng UUID để giữ đúng GPU khi thứ tự device hoặc mask của kernel khác nhau.
BOOT_ENV['CUDA_VISIBLE_DEVICES'] = ','.join(GPU_IDS)
# Giữ LD_LIBRARY_PATH gốc và bổ sung các thư mục driver phổ biến nếu có.
driver_dirs = [p for p in ('/usr/local/nvidia/lib64', '/usr/local/nvidia/lib',
                           '/usr/lib64-nvidia', '/usr/lib/x86_64-linux-gnu')
               if Path(p).is_dir()]
inherited_dirs = [p for p in BOOT_ENV.get('LD_LIBRARY_PATH', '').split(':') if p]
BOOT_ENV['LD_LIBRARY_PATH'] = ':'.join(dict.fromkeys(inherited_dirs + driver_dirs))
VENV = WORK / 'venv_py310_uv'  # Không tái sử dụng venv Python 3.13 tạo dở.
PY = str(VENV / 'bin/python')
subprocess.run([str(UV), '--version'], env=BOOT_ENV, check=True)
# Restore execute permissions before uv tries to inspect an existing Python.
for binary in (WORK / 'python_runtimes').glob('cpython-3.10*/bin/python*'):
    if binary.is_file():binary.chmod(binary.stat().st_mode | 0o111)
for binary in (VENV / 'bin').glob('*'):
    if binary.is_file():binary.chmod(binary.stat().st_mode | 0o111)
subprocess.run([str(UV), 'python', 'install', '3.10'], env=BOOT_ENV, check=True)
if not (VENV / 'pyvenv.cfg').is_file():
    subprocess.run([str(UV), 'venv', '--python', '3.10', '--managed-python', str(VENV)], env=BOOT_ENV, check=True)
# A restored venv may retain pyvenv.cfg but have a broken interpreter symlink.
if not Path(PY).exists() and (VENV / 'pyvenv.cfg').is_file():
    managed = subprocess.check_output([str(UV), 'python', 'find', '--managed-python', '3.10'], env=BOOT_ENV, text=True).strip()
    executable = Path(managed).resolve(strict=True)
    executable.chmod(executable.stat().st_mode | 0o111)
    link = Path(PY)
    link.parent.mkdir(parents=True, exist_ok=True)
    link.unlink(missing_ok=True)
    link.symlink_to(executable)
subprocess.run([PY, '-c', 'import sys; assert sys.version_info[:2] == (3, 10); print(sys.version)'], env=BOOT_ENV, check=True)
# Dùng uv pip trực tiếp, không cần cài pip/ensurepip trong venv.
subprocess.run([str(UV), 'pip', 'install', '--python', PY, '--no-cache',
                '--index-url', 'https://download.pytorch.org/whl/cu124',
                'torch==2.4.1', 'torchvision==0.19.1'], env=BOOT_ENV, check=True)
packages = [
    'numpy==1.26.4','transformers==4.57.1','tokenizers==0.22.1',
    'accelerate==1.11.0','huggingface-hub==0.36.0','safetensors==0.6.2',
    'Pillow==11.3.0','einops==0.8.1','timm==1.0.22','sentencepiece==0.2.0',
    'requests','tqdm','packaging','psutil','protobuf',
]
subprocess.run([str(UV), 'pip', 'install', '--python', PY, '--no-cache', *packages], env=BOOT_ENV, check=True)
ENV = BOOT_ENV.copy()
ENV['PYTHONNOUSERSITE'] = '1'
ENV['HF_HOME'] = str(WORK / 'hf_cache')
ENV['HF_HUB_DISABLE_XET'] = '1'
ENV['OMP_NUM_THREADS'] = '2'
ENV['OPENBLAS_NUM_THREADS'] = '2'
probe = r"""
import ctypes, os, sys
import torch
print('Python:', sys.version.split()[0])
print('torch:', torch.__version__, 'CUDA wheel:', torch.version.cuda)
print('CUDA_VISIBLE_DEVICES:', os.environ.get('CUDA_VISIBLE_DEVICES'))
print('LD_LIBRARY_PATH:', os.environ.get('LD_LIBRARY_PATH'))
try:
    driver = ctypes.CDLL('libcuda.so.1')
    rc = driver.cuInit(0)
    print('NVIDIA driver cuInit result:', rc)
except OSError as exc:
    print('Cannot load NVIDIA driver:', exc)
available = torch.cuda.is_available()
count = torch.cuda.device_count()
print('CUDA available:', available, 'device count:', count)
if not available:
    try:
        torch.cuda.init()
    except Exception as exc:
        print('CUDA initialization error:', type(exc).__name__, str(exc))
    raise SystemExit(2)
if count != 2:
    print('Expected two visible GPUs.')
    raise SystemExit(3)
# Kiểm tra tính toán thật trên từng GPU, không chỉ liệt kê tên.
for i in range(count):
    p = torch.cuda.get_device_properties(i)
    x = torch.ones((8, 8), device=f'cuda:{i}')
    assert (x @ x)[0, 0].item() == 8
    torch.cuda.synchronize(i)
    print(i, p.name, round(p.total_memory / 2**30, 2), 'GiB; compute OK')
import transformers
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, AutoModel, AutoTokenizer
import torchvision, timm
print('transformers:',transformers.__version__,'torchvision:',torchvision.__version__,'timm:',timm.__version__)
"""
probe_result = subprocess.run([PY, '-c', probe], env=ENV, capture_output=True, text=True)
print(probe_result.stdout)
if probe_result.stderr:
    print(probe_result.stderr)
(WORK / 'gpu_check.txt').write_text(
    gpu_query.stdout + '\n' + probe_result.stdout + '\n' + probe_result.stderr,
    encoding='utf-8')
if probe_result.returncode != 0:
    raise RuntimeError('GPU/import check thất bại. Xem CUDA initialization error và gpu_check.txt ở trên. Nếu nvidia-smi không có hai GPU, cần đổi accelerator và khởi động lại session; không cần cài lại package.')
print('Môi trường sẵn sàng; GPU compute check thành công.')
CFG['uv']=str(UV);CONFIG.write_text(json.dumps(CFG,indent=2),encoding='utf-8')
ENV['PYTHONPATH']=str(SCRIPTS)+os.pathsep+str(REPO)
ENV['MPLBACKEND']='Agg'
ENV['CUDA_LAUNCH_BLOCKING']='1' if CFG['sanity'] else '0'
# Check the actual Accelerate hooks before the long native builds.

PREFLIGHT_ENV=ENV.copy()
PREFLIGHT_ENV.pop('HF_HUB_OFFLINE',None)
PREFLIGHT_ENV.pop('TRANSFORMERS_OFFLINE',None)
subprocess.run([PY,'-u',str(SCRIPTS/'apc_qwen_preflight.py'),str(CONFIG)],
               env=PREFLIGHT_ENV,cwd=REPO,check=True)

## Dependency native, weights và renderer

Không dùng nguyên requirements của Orient-Anything vì nó hạ Transformers xuống 4.38. Các package native build trong venv với Torch đã cài, MAX_JOBS=2. Dung lượng model phụ đáng kể; nếu không đủ chỗ, chuẩn bị asset bundle vào Input. Cell này không chạy inference dataset.

In [ ]:
"""Select a CUDA toolkit for the cu124 PyTorch extensions before running native.py."""
import json
import os
import re
import shutil
import subprocess
import tarfile
import tempfile
import urllib.request
from pathlib import Path
import hashlib

def is_cuda124(root):
    compiler = root / 'bin/nvcc'
    if not compiler.is_file() or not os.access(compiler, os.X_OK):
        return False
    version = subprocess.run([str(compiler), '--version'], capture_output=True, text=True)
    return version.returncode == 0 and re.search(r'release 12\.4\b', version.stdout) is not None

def prepare_toolkit(work, env):
    private = work / 'cuda-12.4.1'
    roots = [Path('/usr/local/cuda-12.4'), Path('/usr/local/cuda')]
    current = shutil.which('nvcc', path=env.get('PATH'))
    if current:
        roots.append(Path(current).resolve().parent.parent)
    if (private / '.ready').exists():
        roots.append(private)
    for root in roots:
        if is_cuda124(root) and (root / 'include/cuda_runtime.h').exists() and (root / 'lib64/libcudart.so').exists():
            return root

    base = 'https://developer.download.nvidia.com/compute/cuda/redist/'
    with urllib.request.urlopen(base + 'redistrib_12.4.1.json', timeout=90) as response:
        manifest = json.load(response)
    private.mkdir(exist_ok=True)
    with tempfile.TemporaryDirectory(dir=work) as tmp:
        for package in ('cuda_nvcc', 'cuda_cudart', 'cuda_cccl'):
            item = manifest[package]['linux-x86_64']
            archive = Path(tmp) / (package + '.tar.xz')
            print('CUDA 12.4.1:', package, flush=True)
            digest = hashlib.sha256()
            with urllib.request.urlopen(base + item['relative_path'], timeout=180) as response, archive.open('wb') as dst:
                for chunk in iter(lambda: response.read(1024 * 1024), b''):
                    digest.update(chunk)
                    dst.write(chunk)
            if digest.hexdigest() != item['sha256']:
                raise RuntimeError('CUDA archive checksum mismatch: ' + package)
            unpack = Path(tmp) / package
            with tarfile.open(archive) as tar:
                tar.extractall(unpack, filter='data')
            top, = unpack.iterdir()
            shutil.copytree(top, private, symlinks=True, dirs_exist_ok=True)
    if not (private / 'lib64').exists():
        (private / 'lib64').symlink_to('lib', target_is_directory=True)
    if not is_cuda124(private):
        raise RuntimeError('Private CUDA 12.4 compiler failed to start')
    (private / '.ready').touch()
    return private

CUDA_WORK=Path('/kaggle/temp/apc_cuda_tools')
CUDA_WORK.mkdir(parents=True,exist_ok=True)
toolkit = prepare_toolkit(CUDA_WORK, ENV)
ENV['CUDA_HOME'] = ENV['CUDA_PATH'] = str(toolkit)
ENV['PATH'] = os.pathsep.join([str(toolkit/'bin'), str(Path(PY).parent), ENV['PATH']])
# The Torch cu124 wheels supply the math-library headers used by C++ extensions.
wheel_headers = sorted((Path(PY).parent.parent/'lib/python3.10/site-packages/nvidia').glob('*/include'))
ENV['CPATH'] = os.pathsep.join(filter(None, [str(toolkit/'include'), *map(str,wheel_headers), ENV.get('CPATH')]))
ENV['LIBRARY_PATH'] = os.pathsep.join(filter(None,[str(toolkit/'lib64'),ENV.get('LIBRARY_PATH')]))
ENV['LD_LIBRARY_PATH'] = os.pathsep.join(filter(None,[str(toolkit/'lib64'),ENV.get('LD_LIBRARY_PATH')]))
with tempfile.TemporaryDirectory(dir=WORK) as tmp:
    cu = Path(tmp)/'probe.cu'
    cu.write_text('#include <cuda_runtime.h>\n__global__ void k() {}\nint main(){k<<<1,1>>>(); return cudaDeviceSynchronize()!=cudaSuccess;}')
    exe = Path(tmp)/'probe'
    subprocess.run([str(toolkit/'bin/nvcc'), '-arch=sm_75', str(cu), '-o', str(exe)], env=ENV, check=True)
    subprocess.run([str(exe)], env=ENV, check=True)
print('CUDA 12.4 compile + launch PASS:', toolkit, flush=True)

ENV.pop('HF_HUB_OFFLINE',None);ENV.pop('TRANSFORMERS_OFFLINE',None)
subprocess.run([PY,str(SCRIPTS/'apc_install_native.py'),str(CONFIG)],env=ENV,check=True)
subprocess.run([PY,str(SCRIPTS/'apc_prepare_assets.py'),str(CONFIG),str(WORK/'assets_ready.json')],env=ENV,check=True)
assets=json.loads((WORK/'assets_ready.json').read_text())
ENV['PYTHONPATH']+=os.pathsep+str(REPO/'apc/vision_modules/src/omni3d')+os.pathsep+str(REPO/'apc/vision_modules/src/orient_anything')
subprocess.run([PY,str(SCRIPTS/'apc_kaggle_patches.py'),str(REPO),assets['root'],str(WORK/'patches.json')],env=ENV,check=True)
ENV['HF_HUB_OFFLINE']='1';ENV['TRANSFORMERS_OFFLINE']='1'
# Rendering uses software GLX so no EGL/NVIDIA graphics context is required.
ENV['DISPLAY']=':99';ENV['LIBGL_ALWAYS_SOFTWARE']='1';ENV['MESA_GL_VERSION_OVERRIDE']='3.3';ENV['MPLBACKEND']='Agg'
if not Path('/tmp/.X99-lock').exists():
    display_log=(WORK/'xvfb.log').open('a')
    XVFB=subprocess.Popen(['Xvfb',':99','-screen','0','1280x1024x24','-nolisten','tcp'],stdout=display_log,stderr=subprocess.STDOUT,env=ENV)
    time.sleep(2)
probe="""
import io,json,torch,trimesh
from PIL import Image
from groundingdino import _C
import detectron2,pytorch3d,depth_pro,open3d,segment_anything
from apc_kaggle_memory import plan_qwen_device_map
from apc.apc_pipeline import APC
scene=trimesh.Scene(trimesh.creation.box());raw=scene.save_image(resolution=(128,128),visible=False)
im=Image.open(io.BytesIO(raw));im.verify()
print('Native imports + headless Trimesh render PASS',flush=True)
"""
subprocess.run([PY,'-u','-c',probe],env=ENV,cwd=REPO,check=True)

## Chuẩn bị dữ liệu

Không đưa answer vào prompt. Giữ payload ảnh gốc/flip, khử trùng lặp theo SHA; CircularEval theo qid suffix. Chỉ chọn ba subtype của paper; chuyển visibility trước khi chạy cả hai mode, không đưa answer vào prompt. Giữ source_question/source_choices và question_transform để audit.

In [ ]:
changes={'scripts/apc_experiment.py': [("scratch=Path(cfg['work'])/'scratch'/hashlib.sha256((mode+row['sample_id']).encode()).hexdigest()[:16]", "scratch=folder/'trace'/mode/row['sample_id']"), ("resolved=scratch.resolve();resolved.relative_to((Path(cfg['work'])/'scratch').resolve());shutil.rmtree(resolved)", 'pass  # Keep per-sample evidence'), ('if sanity:result.save', 'if True:result.save'), ("image=image.convert('RGB')", "image=image.convert('RGB')\n            os.environ['APC_TRACE_DIR']=str(scratch)\n            if mode=='apc_vis':image.save(scratch/'input.png')"), ('message=str(exc)', "message=str(exc),traceback=traceback.format_exc(),failed_stage=state.get('stage')")], 'scripts/apc_qwen_adapter.py': [('raw_output=raw[:240]', 'raw_output=raw,prompt=text')], 'apc/apc_pipeline.py': [('import logging', 'import logging\nimport json\nfrom pathlib import Path'), ('for obj_name in objects_of_interest:\n            self.logger.info', 'for obj_index,obj_name in enumerate(objects_of_interest):\n            trace=Path(apc_args["trace_save_dir"])\n            prefix=trace / f"object_{obj_index}"\n            prefix.with_suffix(".name").write_text(obj_name)\n            self.logger.info'), ('if len(boxes) == 0:', "np.save(str(prefix)+'_boxes.npy', np.asarray([np.asarray(b.cpu() if hasattr(b,'cpu') else b) for b in boxes]))\n            if len(boxes) == 0:"), ('# largest mask', "# largest mask\n            Image.fromarray(np.asarray(segment_mask.cpu() if hasattr(segment_mask,'cpu') else segment_mask).astype('uint8')*255).save(str(prefix)+'_mask.png')\n            np.save(str(prefix)+'_selected_box.npy',np.asarray(box2D))\n            if not (trace/'depth.npy').exists():np.save(trace/'depth.npy',depth_npy)"), ('return abstract_scene_tgt', "Path(os.environ['APC_TRACE_DIR'],'scene.json').write_text(json.dumps(dict(viewer=ref_viewer,camera=abstract_scene_src,transformed=abstract_scene_tgt),default=lambda x:np.asarray(x).tolist()))\n        return abstract_scene_tgt")], 'apc/vision_modules/detection.py': [('        boxes_sorted = sort_by_scores', "        from pathlib import Path\n        import json\n        with (Path(os.environ['APC_TRACE_DIR'])/'detector.jsonl').open('a') as stream:\n            stream.write(json.dumps(dict(category=category,boxes=boxes.detach().cpu().tolist(),scores=scores.detach().cpu().tolist()))+'\\n')\n        boxes_sorted = sort_by_scores")], 'apc/vision_modules/orientation.py': [('        direction_vector = self.orientation_to_direction(azimuth_deg, polar_deg)', "        direction_vector = self.orientation_to_direction(azimuth_deg, polar_deg)\n        record['direction_vector']=np.asarray(direction_vector).tolist()\n        (folder/f'orientation_angles_{index:02d}.json').write_text(json.dumps(record,default=lambda x:np.asarray(x).tolist()))"), ('        # Obtain each angle', "        from pathlib import Path\n        import json\n        folder=Path(apc_args['trace_save_dir'])\n        record={'category':category,'bbox':list(bbox) if bbox is not None else None,'angles':np.asarray(pred_angles).tolist()}\n        index=len(list(folder.glob('orientation_input_*.png')))\n        image.save(folder/f'orientation_input_{index:02d}.png')\n        (folder/f'orientation_angles_{index:02d}.json').write_text(json.dumps(record,default=lambda x:np.asarray(x).tolist()))\n        # Obtain each angle")], 'scripts/apc_common.py': [("    if cfg.get('limit') is not None:", "    if cfg.get('paper_tasks'):\n        tasks = set(cfg['paper_tasks'])\n        rows = [r for r in rows if r.get('paper_task') in tasks]\n    if cfg.get('sanity_sample_id') is not None:\n        rows = [r for r in rows if str(r['sample_id']) == str(cfg['sanity_sample_id'])]\n        if len(rows) != 1:\n            raise ValueError('Expected exactly one sanity sample: '+str(cfg['sanity_sample_id']))\n    if cfg.get('limit') is not None:")], 'scripts/apc_score.py': [('Qwen2.5-VL-7B direct/APC-Vis', 'Qwen2.5-VL-7B APC-Vis')]}
for rel, edits in changes.items():
    path=(WORK if rel.startswith('scripts/') else REPO)/rel
    script=path.read_text()
    for before,after in edits:
        if after not in script:
            assert before in script,(rel,before)
            script=script.replace(before,after,1)
    compile(script,str(path),'exec')
    path.write_text(script)
DATA_ENV=ENV.copy();DATA_ENV.pop("HF_HUB_OFFLINE",None);DATA_ENV.pop("TRANSFORMERS_OFFLINE",None)
subprocess.run([PY,str(SCRIPTS/"apc_prepare_data.py"),str(CONFIG)],env=DATA_ENV,cwd=REPO,check=True)
OUT=Path(CFG["output"])
print(json.loads((OUT/"manifest.json").read_text())["selected_rows"],"selected rows")
prepared = json.loads((OUT/'manifest.json').read_text(encoding='utf-8'))
if CFG['limit'] is None:
    assert prepared['selected_rows'] == 692, prepared['selected_rows']


## Chuẩn bị tiến trình và ghi log

In [ ]:
OUT=Path(CFG['output']);manifest=json.loads((OUT/'manifest.json').read_text())
def export_partial():
    import zipfile
    archive=WORK/(CFG['experiment']+'_partial.zip')
    with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
        for path in OUT.rglob('*'):
            if path.is_file() and path.suffix in ('.json','.jsonl','.png','.npy','.name','.log') and 'trace_' not in path.as_posix():z.write(path,path.relative_to(OUT))
        for name in ('memory_plan.json','model_device_map.json','qwen_placement.json','dispatch_preflight.json','pipeline_memory.json','gpu_check.txt'):
            report=WORK/name
            if report.is_file():z.write(report,'runtime_reports/'+name)
    from IPython.display import FileLink,display
    display(FileLink(str(archive)))

def launch(smoke=False):
    folder=OUT/('smoke' if smoke else 'full');folder.mkdir(exist_ok=True)
    state_path=folder/'state.json';state_path.unlink(missing_ok=True)
    command=[PY,'-u',str(SCRIPTS/'apc_experiment.py'),'--manifest',str(OUT/'manifest.json')]
    if smoke:command+=['--smoke']
    before=time.monotonic()
    with (folder/'worker.log').open('a',encoding='utf-8') as log:
        child=subprocess.Popen(command,env=ENV,cwd=REPO,stdout=log,stderr=subprocess.STDOUT)
        failure=None
        try:
            while child.poll() is None:
                try:s=json.loads(state_path.read_text()) if state_path.exists() else {}
                except (ValueError,OSError):s={}
                elapsed=time.monotonic()-before
                rate=s.get('new_processed',0)/max(1,s.get('elapsed_seconds',1))
                remaining=max(0,s.get('total',0)-s.get('processed',0))
                print(('SMOKE' if smoke else 'FULL'),s.get('stage','starting'),s.get('mode',''),s.get('sample_id',''),
                    str(s.get('processed',0))+'/'+str(s.get('total','?')),'errors=',s.get('errors',0),'invalid=',s.get('invalid',0),
                    'elapsed min=',round(elapsed/60,1),'ETA min=',round(remaining/rate/60,1) if rate else '?',flush=True)
                start=s.get('sample_started_unix')
                if start and time.time()-start>CFG['sample_timeout_seconds']+120:
                    failure='Sample watchdog timeout';break
                if not s.get('new_processed') and elapsed>CFG['load_timeout_seconds']+CFG['sample_timeout_seconds']:
                    failure='Pipeline load watchdog timeout';break
                time.sleep(10)
            if failure:raise RuntimeError(failure)
            if child.returncode:
                print('\n'.join((folder/'worker.log').read_text(errors='replace').splitlines()[-70:]))
                raise RuntimeError('APC worker failed; inspect log/state. Results already written are retained.')
        except BaseException:
            if child.poll() is None:child.terminate()
            export_partial();raise
        finally:
            if child.poll() is None:child.terminate()
            try:child.wait(timeout=15)
            except subprocess.TimeoutExpired:child.kill();child.wait()
    print('Finished:',folder)

## Chạy các subtasks của phần này

In [ ]:
launch(smoke=False)


## Chấm điểm và tải báo cáo

Full score giữ mẫu thiếu/lỗi trong mẫu số. Output ZIP chứa prediction, trace ảnh/mask/depth,
cấu hình và báo cáo bố trí bộ nhớ; không chứa token, weights hoặc venv. Giữ ZIP trước khi
kết thúc session. Cùng session có thể chạy lại `launch(smoke=False)` để tiếp tục checkpoint;
`retry_failed=False` giữ các mẫu lỗi đã ghi, đặt True để thử lại chúng.

In [ ]:
subprocess.run([PY,str(SCRIPTS/"apc_score.py"),str(OUT)],env=ENV,cwd=REPO,check=True)

import zipfile
# Keep original/flip images used by every sample.
image_folder=OUT/'images'
image_folder.mkdir(exist_ok=True)
for row in (json.loads(line) for line in (OUT/'samples.jsonl').read_text().splitlines()):
    original=Path(row['image_path'])
    destination=image_folder/original.name
    if not destination.exists():shutil.copy2(original,destination)
for name in ('source_lock.json','patches.json','assets_ready.json','model_device_map.json','memory_plan.json','pipeline_memory.json','gpu_check.txt','dispatch_preflight.json','qwen_placement.json'):
    if (WORK/name).exists():shutil.copy2(WORK/name,OUT/name)
(OUT/'environment.txt').write_text(subprocess.check_output([str(UV),'pip','freeze','--python',PY],env=ENV,text=True),encoding='utf-8')
archive=WORK/(CFG['experiment']+'.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob('*'):
        if p.is_file():z.write(p,p.relative_to(OUT))
from IPython.display import FileLink,display
display(FileLink(str(archive)))